# PepVeg — proteome screening pipeline

Screens 22 phytotherapeutic proteomes for peptides that can occupy the p53-binding
cleft of **MDM2**. It produces the screening figures of the revision of manuscript
**CSBJ-D-26-00633**, *SPAFESTWDILK, a plant-derived dodecapeptide from Zingiber
officinale, as a predicted inhibitor of the MDM2-p53 interaction*; the dynamics,
the MM-PBSA and the other structure predictors are separate and are listed below.

Runs on Colab with a GPU, from top to bottom. It downloads the proteome database
by itself. **No measured value is written into the code**: every figure it reports
is read from an archive at the moment of reporting, and a stage with no archive
says so and computes nothing.

---

## What it does

| stage | | out |
|---|---|---|
| 1 | calibration panel — 6 peptides of known behaviour | measures the 0.80 threshold |
| 2 | 22 proteomes | 337,646 proteins |
| 3 | motif `F-x(3)-W` on whole proteins, before digestion | 60,525 proteins |
| 4 | 6 proteases, Keil rule, 6–20 residues, motif-bearing fragments | 46,626 occurrences |
| 5a | physicochemical pre-filter | 6,602 occurrences left |
| 5b | aggregation by sequence, keeping every origin | **3,214 candidates** |
| 6 | ESM-2 orders all of them, discards none | candidate at rank 17 |
| 7-8 | AlphaFold 3 on the first 30 | **16 above 0.80** · 16 PDB |
| 9 | the candidate: perturbations, alanine scan, p53 mirror | 49 jobs |
| 10 | 3 questions about the method | 30 + 20 + 10 jobs |

The motif is applied at stage 3, before the ranking. Membership of the candidate
set therefore does not depend on any language-model threshold, and no stage
truncates the pool.

---

## What you have to do

Co-folding is not performed here. **6 stages write a job file and read back one
archive**, and the two carry the same name:

| stage | writes | jobs | reads back |
|---|---|---|---|
| 1 | `af3_jobs_calibration_panel.json` | 6 | `af3_results_calibration_panel.zip` |
| 7 | `af3_jobs_canonical_top30.json` | 30 | `af3_results_canonical_top30.zip` |
| 9 | `af3_jobs_candidate_mechanism.json` | 49 | `af3_results_candidate_mechanism.zip` |
| 10 | `af3_jobs_rank_test.json` | 30 | `af3_results_rank_test.zip` |
| 10 | `af3_jobs_alternative_chemistry.json` | 20 | `af3_results_alternative_chemistry.zip` |
| 10 | `af3_jobs_generalisability.json` | 10 | `af3_results_generalisability.zip` |

**How it goes.** Run the notebook from the top. Each of those 6 stages writes
its job file, hands it to you and, if its answer is not in yet, opens a picker.
Take the job file, upload it at [alphafoldserver.com](https://alphafoldserver.com),
wait for every job, select them all and *Download all*. The server returns
`folds_<date>.zip`: rename it to the name in the last column above and give it
to the picker, or drop it into `j_j_af3_archives/` inside the run directory and run
the cell again. What you upload stays with the results and leaves in the export.

Archives are recognised by their **content**: a wrongly named one is still read.
These are the names the deposited archives carry, and they keep the 6 apart in
one folder.

**One decision is yours.** STEP 9 is built around a single candidate, and this
notebook does not choose it: the choice is made on the GROMACS dynamics of the
peptides STEP 8 put above the threshold. Write the peptide into the three
constants at the top of STEP 9, with who decided and on what basis:

```python
CANDIDATE            = "SPAFESTWDILK"
CANDIDATE_DECIDED_BY = "..."
CANDIDATE_BASIS      = "..."
```

Left empty, STEP 9 prints the peptides above the threshold and computes nothing.
The notebook never stops to ask: it only ever waits at a picker.

Coming back to a run you already started, set `START_CLEAN = False` in the Setup
cell, or the digestion of the 60,525 proteins is done again for nothing. The
last cell lists the 6 exchanges and says which ones still have no answer.

---

## What is not done here

5 things are external, each deposited with its own files: **AlphaFold 3
co-folding**, the **molecular dynamics**, the **MM-PBSA**, and the 3 other
structure predictors **Chai-1**, **Protenix** and **EvoEF2**. Archives are read
in place and their sequence alignments are never extracted.

No stage selects a lead. Where a judgement is required — which peptide goes to
long dynamics — the notebook reports the quantities the judgement needs and asks
who decided and on what basis, recording it in the tables.

---

## Requirements, and how to tell it worked

A GPU. ESM-2 runs in float32 with TF32 disabled: similarity differences between
adjacent ranks are of order 1e-6, while TF32 carries about 1e-3, so the ordering
would otherwise depend on which GPU the runtime was assigned.

Each run writes one directory, `run_<database>__<target>`, and inside it one
lettered folder per data reference of the deposit. `START_CLEAN = True` empties
it at start-up.

Four figures identify a faithful re-execution:

| stage | expected |
|---|---|
| 2 | **337,646** proteins loaded from the 22 proteomes |
| 3 | **60,525** proteins carrying the motif |
| 5b | **3,214** distinct candidates |
| 6 | `HAFPELWNLEK` 0.994525 · `FNEIWDLLLETK` 0.993845 · `FESTWDILK` 0.993562, and **`SPAFESTWDILK` at rank 17** of 3,214 |

If any of the four differs, nothing downstream is comparable.

In [ ]:
# ============================================================
# STEP 0 — DEPENDENCY INSTALLATION
# ============================================================
# Pinned versions to avoid the transformers/torch device_mesh
# incompatibility (transformers >=4.45 requires torch >=2.5).
import subprocess, sys

# Versions are pinned only where it matters. transformers and tokenizers are
# left with a lower bound: strict pins have no wheel for Python 3.13, which is
# the interpreter Colab runs, and pip would fail trying to build them.
PINNED = [
    "transformers>=4.45",
    "tokenizers>=0.20",
]
OTHERS = [
    "biopython",
    "pandas",
    "matplotlib",
    "seaborn",
    "tqdm",
    "pyyaml",
]

print("\U0001F4E6 Installing dependencies...")

print("   \U0001F4E5 Installing " + ", ".join(PINNED) + " ...")
# Without -q a pip failure stays readable: with -q it arrives as a bare
# CalledProcessError that does not say what went wrong.
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "--upgrade"] + PINNED
)

for pkg in OTHERS:
    # The module name is not always the package name: biopython imports as Bio.
    ALIAS = {"pyyaml": "yaml", "biopython": "Bio", "scikit-learn": "sklearn"}
    mod = ALIAS.get(pkg, pkg.split("[")[0])
    try:
        __import__(mod)
        print(f"   \u2705 {pkg} already installed")
    except ImportError:
        print(f"   \U0001F4E5 Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        print(f"   \u2705 {pkg} installed")

import torch
print(f"\n   torch {torch.__version__}")
import transformers
print(f"   transformers {transformers.__version__}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"\n\U0001F5A5\uFE0F  GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("\n\u26A0\uFE0F  No GPU detected! Runtime \u2192 Change runtime type \u2192 GPU")

print("\n\u2705 Setup complete!")
print("\u26A0\uFE0F  If you just upgraded transformers, RESTART THE KERNEL before running STEP 6.")


In [ ]:
# ============================================================
# RUN CONFIGURATION - fixed, not asked
# ============================================================
# This notebook reproduces one study: the MDM2 screen on the phytotherapeutic
# proteomes. The target and the database are therefore constants, not questions.

SELECTED_DBS     = ["fitoterapici"]      # 22 proteomes: 15 plants, 7 fungi
SELECTED_TARGETS = ["MDM2"]              # MDM2(25-109), the p53-binding domain

DB_NAMES     = {"fitoterapici": "Phytotherapeutics"}
TARGET_NAMES = {"MDM2": "MDM2"}

print("=" * 58)
print("  DATABASE: %s" % " + ".join(DB_NAMES.get(d, d) for d in SELECTED_DBS))
print("  TARGET:   %s" % ", ".join(TARGET_NAMES.get(t, t) for t in SELECTED_TARGETS))
print("  fixed in the code: this notebook reproduces a single study")
print("=" * 58)

In [ ]:
# ============================================================
# SETUP — ENVIRONMENT AND DATABASE
# ============================================================
import os, glob

if os.path.exists('/workspace'):
    ENV = 'runpod'
    BASE_DIR = '/workspace'
    SAVE_DIR = '/workspace/nutrai_results'
elif os.path.exists('/content'):
    ENV = 'colab'
    BASE_DIR = '/content'
    # Mounting Drive only preserves results between sessions; it is not
    # needed to run the notebook.
    USE_DRIVE = False          # True to save to Drive
    SAVE_DIR = None
    if USE_DRIVE:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        SAVE_DIR = '/content/drive/MyDrive/NutrAI_results'
else:
    ENV = 'local'
    BASE_DIR = os.path.expanduser('~')
    SAVE_DIR = os.path.join(BASE_DIR, 'nutrai_results')

print(f"\U0001F680 {ENV}")

# Every database+target combination has its own directory, and START_CLEAN
# empties it at start-up: two runs never mix.
import shutil

START_CLEAN = True    # False to resume an interrupted run

WORK_DIR = os.path.join(BASE_DIR, "nutrai_pipeline")
DATA_DIR = os.path.join(WORK_DIR, "data")
RUN_ID = "_".join(SELECTED_DBS) + "__" + "_".join(SELECTED_TARGETS)
RESULTS_DIR = os.path.join(WORK_DIR, "run_" + RUN_ID)
DB_PATH = os.path.join(RESULTS_DIR, "nutrai.db")

if START_CLEAN and os.path.exists(RESULTS_DIR):
    shutil.rmtree(RESULTS_DIR)
    print(f"\U0001F9F9 previous run directory removed: {os.path.basename(RESULTS_DIR)}")

for d in [WORK_DIR, DATA_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)
print(f"\U0001F4C1 run: {os.path.basename(RESULTS_DIR)}")
if SAVE_DIR:
    os.makedirs(SAVE_DIR, exist_ok=True)

# The label of the database, for the printout only.
DB_NAMES = {"fitoterapici": "Phytotherapeutics"}

# ── Find and extract tar.gz ──
print("\n\U0001F50D Searching for tar.gz databases...\n")
tar_files = set()
for search in [BASE_DIR, SAVE_DIR or "", WORK_DIR]:
    if search and os.path.exists(search):
        for tf in glob.glob(os.path.join(search, "db_*.tar.gz")):
            tar_files.add(tf)
        for tf in glob.glob(os.path.join(search, "**", "db_*.tar.gz"), recursive=True):
            tar_files.add(tf)

# If nothing is found locally the database is downloaded from the deposit cited
# in the Data availability statement, so anyone can run this notebook.
ZENODO_DB = {
    "fitoterapici": ("db_fitoterapici.tar.gz",
                     "https://zenodo.org/api/records/20447996/files/"
                     "db_fitoterapici.tar.gz/content"),
}
if not tar_files:
    import urllib.request
    for db in SELECTED_DBS:
        entry = ZENODO_DB.get(db)
        if entry is None:
            print(f"   no known deposit for database '{db}'")
            continue
        file_name, url = entry
        dest = os.path.join(WORK_DIR, file_name)
        if not os.path.exists(dest):
            print(f"   downloading {file_name} from Zenodo (about 88 MB)...")
            # It downloads to a temporary name and is renamed only when finished:
            # if the connection drops halfway through the 88 MB, a truncated
            # archive would be taken for a valid one and the download skipped
            # from then on.
            tmp = dest + ".partial"
            try:
                urllib.request.urlretrieve(url, tmp)
                os.replace(tmp, dest)
            except BaseException:
                if os.path.exists(tmp):
                    os.remove(tmp)
                print("   download failed or interrupted: no partial file kept, "
                      "re-run this cell to retry.")
                raise
        tar_files.add(dest)
    if not tar_files:
        raise FileNotFoundError(
            "No database found and no deposit available for this selection. "
            "The phytotherapeutic proteomes are at Zenodo "
            "10.5281/zenodo.20447996.")

FOUND_DBS = []
for tf in sorted(tar_files):
    fname = os.path.basename(tf)
    os.system(f'tar xf "{tf}" -C {DATA_DIR} 2>/dev/null')
    # The archive members carry the nested path workspace/nutrai_pipeline/data/,
    # so the fasta files are moved up to DATA_DIR, which is where the pipeline
    # reads them. The move overwrites: DATA_DIR sits outside the run directory
    # and survives START_CLEAN, so a re-run would otherwise find two copies of
    # each proteome.
    for _f in glob.glob(os.path.join(DATA_DIR, "**", "*.fasta"), recursive=True):
        _dest = os.path.join(DATA_DIR, os.path.basename(_f))
        if os.path.abspath(_f) != os.path.abspath(_dest):
            os.replace(_f, _dest)
    # and the directories the move emptied go too, bottom-up
    for _root, _dirs, _files in os.walk(DATA_DIR, topdown=False):
        if _root != DATA_DIR and not os.listdir(_root):
            os.rmdir(_root)
    # Only the phytotherapeutic archive is used by this notebook. The names are
    # those of the archives on Zenodo, so they are not translated.
    for db_key in ["fitoterapici"]:
        if db_key in fname:
            FOUND_DBS.append(db_key)
            print(f"   \u2705 {DB_NAMES.get(db_key, fname)}: {fname}")
            break

# The database key is the name of the archive on Zenodo, and it is checked here:
# a different key would download a different library.
if SELECTED_DBS != ["fitoterapici"]:
    raise KeyError("SELECTED_DBS = %s. This notebook screens the phytotherapeutic "
                   "proteomes, Zenodo 10.5281/zenodo.20447996." % SELECTED_DBS)

# Counted where the pipeline reads them: the top level of DATA_DIR. A
# recursive count adds any copy left at a nested path, which is not a proteome
# the pipeline will load.
_fasta = glob.glob(os.path.join(DATA_DIR, "*.fasta"))
_nested = [f for f in glob.glob(os.path.join(DATA_DIR, "**", "*.fasta"), recursive=True)
           if os.path.dirname(os.path.abspath(f)) != os.path.abspath(DATA_DIR)]
if not _fasta:
    print("   no .fasta in %s: if STEP 2 finds none either," % DATA_DIR)
    print("   the phytotherapeutic proteomes are at Zenodo 10.5281/zenodo.20447996.")
else:
    print("   %d .fasta in %s" % (len(_fasta), DATA_DIR))
if _nested:
    print("   %d further .fasta remain at nested paths and are NOT loaded, in %s"
          % (len(_nested), ", ".join(sorted({os.path.dirname(f) for f in _nested}))))

# NOTE: SELECTED_TARGETS already set in cell 2 (do not overwrite here)

# ── Summary ──
n_fasta = len(_fasta)
print(f"\n{'='*50}")
print(f"  DATABASE: {' + '.join(DB_NAMES.get(d, d) for d in SELECTED_DBS)}")
print(f"  TARGET:   {', '.join(SELECTED_TARGETS)}")
print(f"  FASTA:    {n_fasta} files")
print(f"{'='*50}")


## Setup — the target

**MDM2**, residues 25–109 of the 1YCR construct: 85 residues, sha1
`5dd9a2fbff5c`, reference peptide p53(17–29). STEP 1 checks that hash before
reading any score.

**`fitoterapici`**: the 22 proteomes listed at STEP 2.

In [ ]:
# ============================================================
# SETUP — MOLECULAR TARGET CONFIGURATION
# ============================================================
import os
import json


# ── MOLECULAR TARGETS ──────────────────────────────────────
# Every field here is read by a later cell:
#   - ref_peptide           the peptide ESM-2 measures similarity against, STEP 6
#   - receptor_seq          the sequence every complex is folded on, STEP 1
#   - receptor_pdb          the crystal the construct comes from
#   - binding_site_residues the cleft, for the site charge of STEP 5a

TARGETS = {
    "MDM2": {
        "name": "MDM2 (p53-binding domain)",
        # p53(17-29), the peptide co-crystallised in PDB 1YCR
        "ref_peptide": "ETFSDLWKLLPEN",
        "receptor_pdb": "1YCR",
        # Leu54, Leu57, Ile61, Met62, Tyr67, Gln72, Val75, Phe86, Phe91, Val93,
        # His96, Ile99, Tyr100, Ile101 - the hydrophobic cleft
        "binding_site_residues": "LLIMYQVFFVHIYI",
        # MDM2(25-109), 85 residues: the construct of the co-crystal and of every
        # complex in this work. sha1 5dd9a2fbff.
        "receptor_seq": ("ETLVRPKPLLLKLLKSVGAQKDTYTMKEVLFYLGQYIMTKRLYDEKQQHIVYCSNDLLGD"
                         "LFGVPSFSVKEHRKIYTMIYRNLVV"),
        "priority": 1,
    },
}
active_targets = {k: v for k, v in TARGETS.items() if k in SELECTED_TARGETS}

for tname, tdata in active_targets.items():
    if tdata["ref_peptide"] is None:
        tdata["ref_peptide"] = tdata["binding_site_residues"]

for tname, tdata in sorted(active_targets.items(), key=lambda x: x[1]["priority"]):
    print("\ntarget      %s(25-109), %d residues"
          % (tname, len(tdata["receptor_seq"])))
    print("crystal     %s" % tdata["receptor_pdb"])
    print("reference   %s   (ESM-2 measures similarity against this)"
          % tdata["ref_peptide"])


## STEP 1 — Calibration panel

Nothing is screened before this: STEP 8 stops if the panel has not been
measured.

6 complexes, all co-folded against **MDM2(25-109)**, 85 residues — the domain
resolved in the p53:MDM2 co-crystal and the construct on which the reference
affinities were measured:

| | class | source of the class |
|---|---|---|
| PMI · 12/1 · p53(17-29) | **binder** | published dissociation constant, 3.3 nM to 483 +/- 79 nM |
| p53 W23A | **non-binder** | established experimentally |
| poly-Ala · scrambled p53 | **designed control** | no measured affinity |

The candidate is not in the panel: it has no measured affinity, so it is
neither a binder nor a non-binder.

This stage **writes the jobs**, you run them, and it reads the archive back.
The statistic is the mean over the 5 models, never best-of-five; the overlap
is checked on the individual models, not on the means.

In [ ]:
# ============================================================
# STEP 1 - CALIBRATION PANEL
# ============================================================
import os, re, json, csv, glob, math, zipfile, hashlib, statistics as st

# The target and its receptor come from the Setup, so the sequence is written in
# one place only. The sha1 is checked here, before any score is read, so that no
# stage can report a number measured on a different construct.
TARGET   = SELECTED_TARGETS[0]
RECEPTOR = active_targets[TARGET]["receptor_seq"]
SHA1_RECEPTOR = "5dd9a2fbff5c"          # MDM2(25-109), 85 residues
assert len(RECEPTOR) == 85, "MDM2(25-109) has 85 residues, this has %d" % len(RECEPTOR)
_sha = hashlib.sha1(RECEPTOR.encode()).hexdigest()
assert _sha.startswith(SHA1_RECEPTOR), (
    "the receptor is not the one every complex in this work was folded on: "
    "sha1 %s instead of %s" % (_sha[:12], SHA1_RECEPTOR))
print("receptor: %s(25-109), %d residues, sha1 %s"
      % (TARGET, len(RECEPTOR), SHA1_RECEPTOR))

AF3_SEED       = "1"
IPTM_THRESHOLD = 0.80     # checked below against the panel, not asserted
# Every stage downstream uses THRESHOLD. It is declared here because this is the
# stage that measures whether it falls inside the empty interval of this target.
THRESHOLD = IPTM_THRESHOLD

try:
    OUT_ROOT = RESULTS_DIR
except NameError:
    OUT_ROOT = "outputs"

# Where the archives you download are put back. Inside the run directory, so
# what you upload sits with the results it explains and leaves in the export.
EXTERNAL = os.environ.get("PEPVEG_EXTERNAL", os.path.join(OUT_ROOT, "j_af3_archives"))
os.makedirs(EXTERNAL, exist_ok=True)
# The stage directories are the directories of the deposit: one per data
# reference, named with the letter the manuscript cites.
OUT_CAL = os.path.join(OUT_ROOT, "c_calibration_panel")
os.makedirs(OUT_CAL, exist_ok=True)

#        sequence           class          Kd nM   note
PANEL = [
    ("TSFAEYWNLLSP",    "binder",           3.3, "PMI"),
    ("MPRFMDYWEGLN",    "binder",         200.0, "12/1"),
    ("ETFSDLWKLLPEN",   "binder",         483.0, "p53(17-29), co-crystal peptide"),
    ("ETFSDLAKLLPEN",   "non-binder",      None, "p53 W23A, established non-binder"),
    ("AAAAAAAAAAAAAAA", "designed control", None, "poly-Ala"),
    ("FWELDSTLKLPNEQ",  "designed control", None, "scrambled p53"),
]
PANEL_CLASS = {seq: (cls, kd, note) for seq, cls, kd, note in PANEL}
REFERENCE_SEQUENCES = set(PANEL_CLASS)

AA3 = {"ALA": "A", "ARG": "R", "ASN": "N", "ASP": "D", "CYS": "C", "GLN": "Q", "GLU": "E",
       "GLY": "G", "HIS": "H", "ILE": "I", "LEU": "L", "LYS": "K", "MET": "M", "PHE": "F",
       "PRO": "P", "SER": "S", "THR": "T", "TRP": "W", "TYR": "Y", "VAL": "V"}


def job_name(raw):
    """A job name the AlphaFold 3 Server accepts.

    The server accepts letters, digits, hyphen and underscore only, and some
    labels carry a slash or parentheses: `MDM2_PANEL_12/1`,
    `MDM2_PANEL_p53(17-29)`. The slash becomes a hyphen, so 12/1 stays readable
    as 12-1; everything else becomes an underscore."""
    out = []
    for ch in raw.replace("/", "-"):
        out.append(ch if (ch.isalnum() or ch in "-_") else "_")
    name = "".join(out)
    while "__" in name:
        name = name.replace("__", "_")
    return name.strip("_")


def af3_job(name, peptide, receptor=None):
    """One AlphaFold 3 Server job, in the format used throughout this work.
    The name passes through job_name, so every call site is sanitised the same
    way."""
    return {"name": job_name(name), "modelSeeds": [AF3_SEED],
            "sequences": [
                {"proteinChain": {"sequence": receptor or RECEPTOR, "count": 1,
                                  "useStructureTemplate": True}},
                {"proteinChain": {"sequence": peptide, "count": 1,
                                  "useStructureTemplate": True}}],
            "dialect": "alphafoldserver", "version": 3}


def chains_of(model_text):
    """Sequence of every chain, from the CA atoms."""
    cols, inloop, seen = [], False, {}
    for line in model_text.split("\n"):
        s = line.strip()
        if s.startswith("_atom_site."):
            cols.append(s.split(".", 1)[1]); inloop = True; continue
        if inloop and cols:
            if not s or s.startswith("#") or s.startswith("loop_") or s.startswith("_"):
                if seen:
                    break
                continue
            f = s.split()
            if len(f) < len(cols):
                continue
            d = dict(zip(cols, f))
            if d.get("label_atom_id") == "CA":
                seen.setdefault(d.get("label_asym_id"), []).append(
                    AA3.get(d.get("label_comp_id"), "X"))
    return {k: "".join(v) for k, v in seen.items()}


def complexes_in(archive):
    """Peptide, receptor length and the five iPTM values of every job in an AF3
    archive. Alignments and templates are never read: they are large and unused."""
    z = zipfile.ZipFile(archive)
    jobs = {}
    for n in z.namelist():
        if "/msas/" in n or "/templates/" in n or n.endswith("/"):
            continue
        key = "/".join(n.split("/")[:-1]) or n
        if "summary_confidences_" in n and n.endswith(".json"):
            jobs.setdefault(key, {}).setdefault("iptm", []).append(
                json.loads(z.read(n).decode("utf8"))["iptm"])
        elif re.search(r"model_0\.cif$", n) or re.search(r"modello_migliore\.cif$", n) or re.search(r"best_model\.cif$", n):
            jobs.setdefault(key, {})["model"] = n
    # A directory nested inside a job - the five models kept apart - has its own
    # summary_confidences files and would be read as a second complex.
    nested = {k for k in jobs for j in jobs if k != j and k.startswith(j + "/")}
    out = []
    for key, d in jobs.items():
        if key in nested or "iptm" not in d or "model" not in d:
            continue
        ch = chains_of(z.read(d["model"]).decode("utf8", "replace"))
        if len(ch) < 2:
            continue
        out.append({"peptide": min(ch.values(), key=len),
                    "receptor_len": len(max(ch.values(), key=len)),
                    "iptm": d["iptm"], "job": key,
                    "archive": os.path.basename(archive),
                    "zip": archive, "model": d["model"]})
    return out


def effect_size(a, b):
    """Cohen's d, Hedges' g and the 95% CI with df-weighted pooled SD, as declared."""
    n1, n2 = len(a), len(b)
    sp = math.sqrt(((n1 - 1) * st.variance(a) + (n2 - 1) * st.variance(b)) / (n1 + n2 - 2))
    d = (st.mean(a) - st.mean(b)) / sp
    J = 1 - 3 / (4 * (n1 + n2) - 9)
    se = math.sqrt((n1 + n2) / (n1 * n2) + d * d / (2 * (n1 + n2 - 2)))
    return {"n1": n1, "n2": n2, "pooled_sd": sp, "d": d, "g": d * J, "J": J,
            "se": se, "lo": d - 1.96 * se, "hi": d + 1.96 * se}


def expected_archive(jobs_path):
    """The archive that answers a job file: af3_jobs_X.json -> af3_results_X.zip.

    The name by which a folder of six archives is read without opening any.
    Recognition is by content, so a differently named archive is still read."""
    b = os.path.basename(jobs_path)
    if b.startswith("af3_jobs_") and b.endswith(".json"):
        return "af3_results_" + b[len("af3_jobs_"):-len(".json")] + ".zip"
    return os.path.splitext(b)[0] + ".zip"


def jobs_of_archive(archive):
    """The job file an archive answers: af3_results_X.zip -> af3_jobs_X.json."""
    if not archive:
        return None
    b = os.path.basename(archive)
    if b.startswith("af3_results_") and b.endswith(".zip"):
        return "af3_jobs_" + b[len("af3_results_"):-len(".zip")] + ".json"
    return None


def archives_behind(complexes):
    """Which archive each complex came from, and how many from each."""
    count = {}
    for c in complexes:
        count[c["archive"]] = count.get(c["archive"], 0) + 1
    return sorted(count.items())


# Six stages register their exchange here; the end of the run lists them.
EXCHANGES = []


def deliver(path, needed=True, n_jobs=None):
    """Hands the job file over, whether or not its archive is already here.

    The job file states which complexes were asked for, on which receptor and
    with which seed, so all six are needed to re-run this notebook from an
    empty folder. `needed=False` changes the wording only.

    The download is reported as an attempt: Chrome blocks automatic downloads
    after the first few from one tab, and files.download() raises nothing when
    that happens.
    """
    _kb = os.path.getsize(path) / 1024
    _zip = expected_archive(path)
    EXCHANGES.append({"job": os.path.basename(path), "archive": _zip,
                      "n_jobs": n_jobs, "to_run": bool(needed), "path": path})
    print("\n   FILE  %s   (%.1f kB)%s"
          % (path, _kb, "" if n_jobs is None else "   %d jobs" % n_jobs))
    print("   -> run on alphafoldserver.com, rename the result %s," % _zip)
    print("      put it in '%s/'%s"
          % (EXTERNAL, "" if needed else "   [already there: nothing to run]"))
    try:
        from google.colab import files
    except Exception:
        print("   Not on Colab: take the file from the path above.")
        return
    try:
        files.download(path)
        print("   (download attempted; if it did not arrive, take it from the "
              "Files pane)")
    except Exception as _e:
        print("   (download refused, %s; take it from the Files pane)"
              % type(_e).__name__)


def exchange_summary():
    """The six exchanges with the server, and the state of each."""
    if not EXCHANGES:
        print("No stage asked for co-folding in this run.")
        return
    # The state is read from EXTERNAL now, not remembered from when the stage
    # wrote its job file: between the two an archive may have been loaded, and
    # a summary printed at the end must say what is on disk at the end.
    here = {os.path.basename(z) for z in glob.glob(os.path.join(EXTERNAL, "*.zip"))}
    pool, _ = complexes_by_sequence(verbose=False)
    answered = {c["archive"] for c in pool.values()}
    for s_ in EXCHANGES:
        s_["to_run"] = s_["archive"] not in here and s_["archive"] not in answered
    to_run = [s for s in EXCHANGES if s["to_run"]]
    print("\n" + "=" * 70)
    print("THE EXCHANGES WITH THE ALPHAFOLD 3 SERVER")
    print("=" * 70)
    print("%-42s%7s  %s" % ("job file handed over", "jobs", "state"))
    for s_ in EXCHANGES:
        print("%-42s%7s  %s"
              % (s_["job"], s_["n_jobs"] if s_["n_jobs"] is not None else "-",
                 "to run" if s_["to_run"] else "answered"))
    print("\n%d job file%s written and offered for download."
          % (len(EXCHANGES), " was" if len(EXCHANGES) == 1 else "s were"))
    print("If one did not reach your Downloads folder the browser blocked it:")
    print("take it from the Files pane, at the path its own stage printed.")
    if to_run:
        print("\n%d have no archive in '%s' yet:" % (len(to_run), EXTERNAL))
        for s_ in to_run:
            print("   %-42s -> %s" % (s_["job"], s_["archive"]))
    else:
        print("\nEvery stage already has its archive in '%s'." % EXTERNAL)


_POOL_CACHE = {}


def complexes_by_sequence(verbose=True):
    """Every complex in every archive in EXTERNAL, indexed by its peptide.

    Archives are matched by CONTENT, never by name: the server returns them as
    folds_<date>.zip, and several sit in this folder at once.

    Returns (pool, duplicates). A peptide folded twice appears in duplicates:
    two runs are two measurements, and the stage that reads it says which one
    it used."""
    zips = sorted(glob.glob(os.path.join(EXTERNAL, "*.zip")))
    key = tuple((z, os.path.getmtime(z), os.path.getsize(z)) for z in zips)
    if _POOL_CACHE.get("key") == key:
        return _POOL_CACHE["pool"], _POOL_CACHE["dupes"]

    pool, dupes = {}, {}
    for zp in zips:
        for c in complexes_in(zp):
            if len(c["iptm"]) < 2:
                continue
            # Two runs of one peptide are two measurements: both are
            # reported and the first is used.
            if c["peptide"] in pool:
                dupes.setdefault(c["peptide"], [pool[c["peptide"]]["job"]]).append(c["job"])
            else:
                pool[c["peptide"]] = c
    _POOL_CACHE.update(key=key, pool=pool, dupes=dupes)
    # Printed whenever the set of archives changes, whatever `verbose` says,
    # so no stage reads an archive without naming it.
    before = {z for z, _m, _s in _POOL_CACHE.get("shown") or ()}
    first = _POOL_CACHE.get("shown") is None
    changed = _POOL_CACHE.get("shown") != key
    if verbose or changed:
        _POOL_CACHE["shown"] = key
        if not zips:
            print("   no AF3 archive in '%s' yet" % EXTERNAL)
        else:
            # Counted on each archive's own contents, not on its
            # contribution to the deduplicated pool.
            n_per_zip = {}
            for c in [x for z_ in zips for x in complexes_in(z_)]:
                n_per_zip[c["archive"]] = n_per_zip.get(c["archive"], 0) + 1
            # The first time the whole folder is listed; after that only what
            # has just arrived, with the running total.
            show = zips if (first or verbose) else [z for z in zips if z not in before]
            # The total is the sum of the column below; distinct peptides
            # are fewer when a peptide sits in two archives.
            _tot = sum(n_per_zip.values())
            print("   %d AF3 archive(s) in '%s', %d complexes%s%s:"
                  % (len(zips), EXTERNAL, _tot,
                     "" if _tot == len(pool) else " for %d distinct peptides" % len(pool),
                     "" if len(show) == len(zips) else ", new here"))
            for z in show:
                b = os.path.basename(z)
                print("      %-46s %4d complexes  %6.1f MB"
                      % (b, n_per_zip.get(b, 0), os.path.getsize(z) / 1e6))
    return pool, dupes


def upload_archives(what, expect=None):
    """Opens a file picker in this cell and keeps whatever is chosen.

    The file is saved into EXTERNAL under its own name and recognised by
    content. A large archive can also be dropped into that folder from the
    Files pane and the cell re-run."""
    _jobs = jobs_of_archive(expect)
    _da = "" if not _jobs else " of %s" % _jobs
    print("\n   >>> INSERT HERE THE AF3 RESULTS%s <<<" % _da.upper())
    print("   the archive the AlphaFold 3 Server returned for those jobs: %s"
          % (what))
    if expect:
        print("   rename it %s and put it in '%s/'." % (expect, EXTERNAL))
    try:
        from google.colab import files
    except Exception:
        print("   Not on Colab: put it in '%s' and run this cell again."
              % EXTERNAL)
        return []
    print("   Choose it here:")
    try:
        chosen = files.upload()
    except Exception as _e:
        print("   upload cancelled or failed (%s)." % type(_e).__name__)
        return []
    saved = []
    for _name, _data in chosen.items():
        _dest = os.path.join(EXTERNAL, os.path.basename(_name))
        with open(_dest, "wb") as _f:
            _f.write(_data)
        saved.append(_dest)
        print("   saved %s  (%.1f MB)" % (_dest, len(_data) / 1e6))
    return saved


def take(sequences, what, expect=None):
    """The complexes for these sequences, without stopping the run.

    Offers the picker once, reports what is still missing, and returns. STEP 1
    uses this: STEPs 2 to 7 do not read the calibration, so a missing panel
    archive does not have to stop them."""
    sequences = list(sequences)
    pool, dupes = complexes_by_sequence()
    absent = [s for s in sequences if s not in pool]
    if absent:
        print("   %s: %d of %d missing" % (what, len(absent), len(sequences)))
        if upload_archives(what, expect):
            pool, dupes = complexes_by_sequence(verbose=False)
            absent = [s for s in sequences if s not in pool]
    got = {s: pool[s] for s in sequences if s in pool}
    twice = {s: dupes[s] for s in sequences if s in dupes}
    if absent:
        print("   still missing: %s" % ", ".join(absent[:8]))
        print("   this stage computes nothing without them, and the stages that")
        print("   use its result will stop until it has been run.")
    else:
        print("   %d of %d complexes read" % (len(got), len(sequences)))
    for a_, n_ in archives_behind(got.values()):
        print("      %d of them from %s" % (n_, a_))
    for s, j in twice.items():
        print("   ANOMALY: %s appears in %d runs (%s). The first is used; a"
              % (s, len(j), ", ".join(j)))
        print("   screen measures every peptide once, so an archive should carry")
        print("   one run per peptide.")
    return got, absent, twice


def require(sequences, what, expect=None):
    """The complexes for these sequences, or the pipeline stops.

    If any is missing the cell asks for the archive once, then raises. Nothing
    is computed on partial input."""
    sequences = list(sequences)
    for _attempt in (1, 2):
        pool, dupes = complexes_by_sequence(verbose=(_attempt == 1))
        got = {s: pool[s] for s in sequences if s in pool}
        absent = [s for s in sequences if s not in pool]
        if not absent:
            twice = {s: dupes[s] for s in sequences if s in dupes}
            if twice:
                print("   folded more than once - two runs are two measurements:")
                for s_, j in twice.items():
                    print("      %-16s %d runs (%s); the first is used"
                          % (s_, len(j), ", ".join(j)))
            print("   %d of %d complexes read" % (len(got), len(sequences)))
            for a_, n_ in archives_behind(got.values()):
                print("      %d of them from %s" % (n_, a_))
            return got, twice
        print("   %s: %d of %d not in any archive yet"
              % (what, len(absent), len(sequences)))
        if _attempt == 1:
            upload_archives(what, expect)

    raise FileNotFoundError(
        "STOP - %s: %d of %d peptides are not in any archive in '%s'.\n\n"
        "missing: %s\n\n"
        "Run the job file this notebook wrote on the AlphaFold 3 Server, then\n"
        "re-run this cell and choose the archive it returns%s.\n"
        "The file name does not matter: archives are read by content."
        % (what, len(absent), len(sequences), EXTERNAL,
           ", ".join(absent[:12]) + (" ..." if len(absent) > 12 else ""),
           "" if not expect else " (%s)" % expect))


# ── write the jobs ──────────────────────────────────────────────────────────
jobs, manifest = [], []
for seq, cls, kd, note in PANEL:
    jobs.append(af3_job("%s_PANEL_%s" % (TARGET, note.split(",")[0].replace(" ", "_")), seq))
    manifest.append({"peptide": seq, "length": len(seq), "class": cls,
                     "kd_nM": kd, "note": note})

p_jobs = os.path.join(OUT_CAL, "af3_jobs_calibration_panel.json")
p_man = os.path.join(OUT_CAL, "calibration_panel_manifest.csv")
with open(p_jobs, "w") as f:
    json.dump(jobs, f, indent=1)
with open(p_man, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(manifest[0].keys()))
    w.writeheader(); w.writerows(manifest)

print("calibration panel: %d complexes on %s(25-109), seed %s" % (len(jobs), TARGET, AF3_SEED))
print("%-17s%-18s%9s  %s" % ("peptide", "class", "Kd nM", "note"))
for m in manifest:
    print("%-17s%-18s%9s  %s"
          % (m["peptide"], m["class"], "-" if m["kd_nM"] is None else "%.1f" % m["kd_nM"], m["note"]))
print("\nthe candidate is not in the panel: it has no measured affinity, so it is")
print("neither a binder nor a non-binder and cannot be part of either class.")
print("\nwritten:\n   %s\n   %s" % (p_jobs, p_man))

# ── read the archive back, if it is there ───────────────────────────────────
print("\nreading the panel back:")
_got, _absent, _ = take(list(PANEL_CLASS), "calibration panel",
                        "af3_results_calibration_panel.zip")
if _absent:
    print("\nRun these %d jobs on AlphaFold 3 Server, then put the archive in"
          % len(jobs))
    print("'%s' and run this cell again." % EXTERNAL)
deliver(p_jobs, needed=bool(_absent), n_jobs=len(jobs))
CALIBRATION = None
if _got:
    rows = []
    for c in _got.values():
        cls, kd, note = PANEL_CLASS[c["peptide"]]
        rows.append({"peptide": c["peptide"], "class": cls, "kd_nM": kd, "note": note,
                     "iptm_mean": st.mean(c["iptm"]),
                     "iptm_sd": st.stdev(c["iptm"]) if len(c["iptm"]) > 1 else 0.0,
                     "iptm_models": c["iptm"], "receptor_len": c["receptor_len"],
                     "archive": c["archive"], "job": c["job"]})
    missing = _absent
    odd = [r for r in rows if r["receptor_len"] != len(RECEPTOR)]
    if odd:
        print("   ANOMALY: complexes on another construct: %s"
              % ", ".join("%s (%d residues)" % (r["peptide"], r["receptor_len"]) for r in odd))

    if not missing:
        rows.sort(key=lambda r: -r["iptm_mean"])
        print("\n%-17s%-18s%8s%8s%5s  models" % ("peptide", "class", "iPTM", "SD", "n"))
        for r in rows:
            print("%-17s%-18s%8.3f%8.4f%5d  %s"
                  % (r["peptide"], r["class"], r["iptm_mean"], r["iptm_sd"],
                     len(r["iptm_models"]), " ".join("%.2f" % v for v in sorted(r["iptm_models"], reverse=True))))

        bind = [r for r in rows if r["class"] == "binder"]
        neg = [r for r in rows if r["class"] != "binder"]
        lo_b = min(r["iptm_mean"] for r in bind)
        hi_n = max(r["iptm_mean"] for r in neg)
        gap = lo_b - hi_n
        model_gap = min(min(r["iptm_models"]) for r in bind) - max(max(r["iptm_models"]) for r in neg)

        print("\nminimum gap on the means      %+.3f" % gap)
        print("gap on individual models      %+.3f   %s"
              % (model_gap, "zero overlap" if model_gap > 0 else "THE MODELS OVERLAP"))
        print("threshold %.2f                 %s"
              % (IPTM_THRESHOLD,
                 ("inside the empty interval %.3f - %.3f" % (hi_n, lo_b))
                 if hi_n < IPTM_THRESHOLD < lo_b else
                 ("NOT inside the empty interval %.3f - %.3f - it must be justified or changed"
                  % (hi_n, lo_b))))

        THRESHOLD = IPTM_THRESHOLD      # validated on the panel just above
        CALIBRATION = {"target": TARGET, "receptor_len": len(RECEPTOR),
                       "threshold": IPTM_THRESHOLD, "gap": gap, "model_gap": model_gap,
                       "lowest_binder": lo_b, "highest_negative": hi_n,
                       "binders": [r["iptm_mean"] for r in bind],
                       "reference_sequences": sorted(REFERENCE_SEQUENCES),
                       "panel": rows}

        t = os.path.join(OUT_CAL, "table_calibration_panel.csv")
        with open(t, "w", newline="") as f:
            w = csv.writer(f)
            w.writerow(["peptide", "class", "kd_nM", "iptm_mean", "iptm_sd",
                        "iptm_models", "receptor_len", "note", "archive", "job"])
            for r in rows:
                w.writerow([r["peptide"], r["class"], r["kd_nM"], round(r["iptm_mean"], 4),
                            round(r["iptm_sd"], 4),
                            " ".join("%.2f" % v for v in sorted(r["iptm_models"], reverse=True)),
                            r["receptor_len"], r["note"], r["archive"], r["job"]])
        s = os.path.join(OUT_CAL, "calibration_panel_summary.txt")
        with open(s, "w") as f:
            f.write("target                %s(25-109), %d residues\n" % (TARGET, len(RECEPTOR)))
            f.write("statistic             mean over the five models\n")
            f.write("complexes             %d (%d binders with published Kd, %d negatives)\n"
                    % (len(rows), len(bind), len(neg)))
            f.write("minimum gap           %+.3f (means)   %+.3f (individual models)\n" % (gap, model_gap))
            f.write("threshold adopted     iPTM >= %.2f\n" % IPTM_THRESHOLD)
            f.write("empty interval        %.3f - %.3f\n" % (hi_n, lo_b))
        print("\nwritten:\n   %s\n   %s" % (t, s))

## STEP 2 — The proteomes

22 phytotherapeutic proteomes, 15 plants and 7 fungi, downloaded by the Setup
cell from Zenodo [10.5281/zenodo.20447996](https://doi.org/10.5281/zenodo.20447996),
88 MB. The stage writes `proteomes_used.csv`: species, class,
UniProt reference proteome and the protein count it measured for each. 10 of
the 22 have no reference proteome, come from the corresponding
UniProtKB queries, and carry `N/A` in that column.

337,646 proteins is the control figure of the whole re-execution: every count
downstream is a fraction of it.

In [ ]:
# ============================================================
# STEP 2 - PROTEOMES: THE LIBRARY AND ITS PROVENANCE
# ============================================================
# species -> (class, UniProt reference proteome)
# The binomial is the identifier: it needs no translation and it is what the
# Methods section cites.
PROTEOMES = {
    "Artemisia_annua":             ("plant", "UP000245207"),
    "Astragalus_membranaceus":     ("plant", "N/A"),
    "Camellia_sinensis":           ("plant", "UP000306102"),
    "Centella_asiatica":           ("plant", "N/A"),
    "Cinnamomum_verum":            ("plant", "N/A"),
    "Cordyceps_militaris":         ("fungus", "UP000001610"),
    "Crocus_sativus":              ("plant", "N/A"),
    "Curcuma_longa":               ("plant", "N/A"),
    "Ganoderma_lucidum":           ("fungus", "N/A"),
    "Glycyrrhiza_glabra":          ("plant", "N/A"),
    "Grifola_frondosa":            ("fungus", "UP000092993"),
    "Lentinula_edodes":            ("fungus", "UP000188533"),
    "Moringa_oleifera":            ("plant", "N/A"),
    "Olea_europaea":               ("plant", "UP000594638"),
    "Ophiocordyceps_sinensis":     ("fungus", "UP000557566"),
    "Panax_ginseng":               ("plant", "N/A"),
    "Pleurotus_ostreatus":         ("fungus", "UP000623687"),
    "Reynoutria_japonica":         ("plant", "N/A"),
    "Silybum_marianum":            ("plant", "N/A"),
    "Trametes_versicolor":         ("fungus", "UP000054317"),
    "Vanilla_planifolia":          ("plant", "UP000636800"),
    "Zingiber_officinale":         ("plant", "N/A"),
}

active_proteomes = {k: {"name": k.replace("_", " "), "class": v[0],
                        "proteome_id": v[1]}
                    for k, v in PROTEOMES.items()}

VALID_AA = set("ACDEFGHIKLMNPQRSTVWY")


def parse_fasta(filepath):
    """FASTA parser. Yields (header, sequence)."""
    header, seq_parts = "", []
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if line.startswith(">"):
                if header:
                    yield header, "".join(seq_parts)
                header = line[1:]
                seq_parts = []
            else:
                seq_parts.append(line.upper())
    if header:
        yield header, "".join(seq_parts)


# ── count what is on disk ───────────────────────────────────────────────────
# the Setup cell downloaded, extracted and gathered the fasta files into DATA_DIR.
proteome_stats = {}
for pname in active_proteomes:
    fpath = os.path.join(DATA_DIR, f"{pname}.fasta")
    if not os.path.exists(fpath):
        continue
    n_seq = sum(1 for line in open(fpath) if line.startswith(">"))
    if n_seq:
        proteome_stats[pname] = n_seq
    else:
        print(f"   empty file: {pname}")

_extra = sorted({os.path.basename(f)[:-6]
                 for f in glob.glob(os.path.join(DATA_DIR, "*.fasta"))}
                - set(active_proteomes))
if _extra:
    print(f"   {len(_extra)} fasta in the data directory are not among the 22 "
          f"and are ignored: {', '.join(_extra[:6])}")

missing = [k for k in active_proteomes if k not in proteome_stats]
if missing:
    print(f"   MISSING {len(missing)} of {len(active_proteomes)} proteomes:")
    for m in missing:
        print(f"      {m}")
if not proteome_stats:
    raise FileNotFoundError(
        f"no proteome fasta found in {DATA_DIR}. The archive is at Zenodo "
        "10.5281/zenodo.20447996 and the Setup cell retrieves it.")

_plants = sum(1 for k in proteome_stats if active_proteomes[k]["class"] == "plant")
_fungi = sum(1 for k in proteome_stats if active_proteomes[k]["class"] == "fungus")
total_proteins = sum(proteome_stats.values())

print(f"{len(proteome_stats)} proteomes loaded - {_plants} plants, {_fungi} fungi")
print(f"{total_proteins:,} proteins")

# ── the provenance file ─────────────────────────────────────────────────────
_prov = os.path.join(RESULTS_DIR, "a_proteomes_and_library", "proteomes_used.csv")
os.makedirs(os.path.dirname(_prov), exist_ok=True)
with open(_prov, "w", newline="") as _f:
    _w = csv.writer(_f)
    _w.writerow(["species", "class", "uniprot_proteome", "proteins_loaded"])
    for k in sorted(proteome_stats, key=lambda k: -proteome_stats[k]):
        _w.writerow([k.replace("_", " "), active_proteomes[k]["class"],
                     active_proteomes[k]["proteome_id"], proteome_stats[k]])
print(f"written: {_prov}")

## STEP 3 — Pharmacophore filter on the proteomes

The motif is applied here, to whole proteins, before anything is digested.

The query is the anchor of p53 itself: a phenylalanine and a tryptophan four
residues apart, `F-x(3)-W`. In p53(17–29) these are Phe19 and Trp23, and with
Leu26 they fill the 3 sub-pockets of the MDM2 cleft.

The spacing is measured at STEP 9, on sequences of **identical composition**.
Moving the tryptophan one residue either way costs the candidate 0.27 of
interface confidence: 0.834 against 0.556 one residue earlier and 0.572 one
residue later, means over the 5 models.

A peptide carrying `F-x(3)-W` is a substring of the protein it was cut from, so
that protein carries the motif too. A protein without it therefore **cannot**
produce a candidate, and setting it aside cannot lose one.

This stage writes one filtered FASTA per proteome and reports, per proteome,
how many proteins carry the motif. **STEP 4 digests those files.**

In [ ]:
# ============================================================
# STEP 3 - PHARMACOPHORE FILTER ON THE PROTEOMES
# ============================================================
# One filtered FASTA per proteome, holding only the proteins that carry the
# motif. STEP 4 digests these files.
import re as _re_module

MOTIF = r"F.{3}W"            # the canonical i/i+4 Phe-Trp spacing
_motif_rx = _re_module.compile(MOTIF)

MOTIF_DIR = os.path.join(DATA_DIR, "motif_only")
os.makedirs(MOTIF_DIR, exist_ok=True)

motif_proteomes = {}        # proteome -> proteins kept
motif_fasta = {}            # proteome -> path of the file STEP 4 reads

print(f"keeping only proteins that contain {MOTIF}\n")
print(f"{'proteome':28s}{'proteins':>10s}{'with motif':>12s}{'':>3s}{'share':>7s}")
_tot_in = _tot_out = 0
for pname, pdata in active_proteomes.items():
    src = os.path.join(DATA_DIR, f"{pname}.fasta")
    if not os.path.exists(src):
        continue
    dst = os.path.join(MOTIF_DIR, f"{pname}.fasta")
    n_in = n_out = 0
    with open(dst, "w") as f_out:
        for header, seq in parse_fasta(src):
            n_in += 1
            clean = "".join(c for c in seq if c in VALID_AA)
            if not _motif_rx.search(clean):
                continue
            n_out += 1
            f_out.write(f">{header}\n")
            for _i in range(0, len(seq), 60):
                f_out.write(seq[_i:_i + 60] + "\n")
    motif_proteomes[pname] = n_out
    motif_fasta[pname] = dst
    _tot_in += n_in
    _tot_out += n_out
    print(f"{pdata['name'][:28]:28s}{n_in:>10,}{n_out:>12,}{'':>3s}"
          f"{100 * n_out / max(1, n_in):>6.1f}%")

print(f"\n{'total':28s}{_tot_in:>10,}{_tot_out:>12,}{'':>3s}"
      f"{100 * _tot_out / max(1, _tot_in):>6.1f}%")
print(f"\n{_tot_in - _tot_out:,} proteins are not digested at all: they cannot")
print(f"produce a peptide carrying {MOTIF}, because such a peptide is a")
print("substring of its protein. The candidate set is unchanged by this.")
print(f"\nwritten: {MOTIF_DIR}")

## STEP 4 — In silico enzymatic hydrolysis

Cleavage is simulated with **6 real enzymes**:

| enzyme | cleaves after |
|---|---|
| Trypsin | K, R |
| Pepsin | F, Y, W, L |
| Chymotrypsin | F, Y, W |
| Papain | R, K, Q, H, G, Y, F, W |
| Bromelain | K, R, A, Y |
| Alcalase | F, W, Y, L, I, V, M, A |

Cleavage happens after each listed residue **except where a proline follows**
(Keil rule): endopeptidases do not hydrolyse the X–P bond. Missed cleavages are
not modelled.

Fragments of **6 to 20 residues** are retained, and only if they contain at
least one aromatic residue (F, W, Y, H) **and** one basic one (R, K, H). Each
record lists **every** enzyme and **every** proteome that generates the
fragment, not only the first encountered.

The proteins come from **STEP 3**: only those carrying the motif are here at
all. A second filter applies to the **fragments**, because of a motif-bearing
protein most fragments do not carry the motif — the cleavage falls inside it, or
the fragment falls outside it. What reaches disk *is* the candidate pool.

In [ ]:
# ============================================================
# STEP 4 - IN SILICO ENZYMATIC HYDROLYSIS, SIX PROTEASES
# ============================================================
import csv
from tqdm.auto import tqdm
from collections import defaultdict

# Enzyme specificities: the enzyme cleaves AFTER these residues
ENZYMES = {
    "Trypsin":      set("KR"),
    "Pepsin":       set("FYWL"),
    "Chymotrypsin": set("FYW"),
    "Papain":       set("RKQHGYFW"),
    "Bromelain":    set("KRAY"),
    "Alcalase":     set("FWYLIVMA"),
}

MIN_LEN = 6
MAX_LEN = 20
BLOCK_CLEAVAGE_BEFORE_PRO = True   # Keil rule: no cleavage before a proline
AROMATIC = set("FWYH")
BASIC = set("RKH")

def enzymatic_digest(protein_seq, enzyme_cuts):
    """
    Simulate enzymatic cleavage of a protein.
    The enzyme cleaves AFTER the specified residues.
    Returns a list of (peptide, start_pos).
    """
    fragments = []
    start = 0
    for i, aa in enumerate(protein_seq):
        if aa in enzyme_cuts:
            # A proline in P1' blocks cleavage (Keil 1992). Without this check the
            # digestion produces fragments that never form in reality.
            if BLOCK_CLEAVAGE_BEFORE_PRO and protein_seq[i+1:i+2] == "P":
                continue
            frag = protein_seq[start:i+1]
            if MIN_LEN <= len(frag) <= MAX_LEN:
                fragments.append((frag, start))
            start = i + 1
    frag = protein_seq[start:]
    if MIN_LEN <= len(frag) <= MAX_LEN:
        fragments.append((frag, start))
    return fragments

def passes_biofilter(peptide):
    """Biochemical filter: at least 1 aromatic AND 1 basic, standard aa only."""
    if not all(aa in VALID_AA for aa in peptide):
        return False
    has_aromatic = any(aa in AROMATIC for aa in peptide)
    has_basic = any(aa in BASIC for aa in peptide)
    return has_aromatic and has_basic

PEPTIDES_CSV = os.path.join(RESULTS_DIR, "peptides_hydrolysis.csv")
RECORD_ALL_OCCURRENCES = True   # one row per occurrence, aggregated at STEP 5

# The proteins come from STEP 3, already filtered. A second filter applies
# here, to the fragments: of a motif-bearing protein most fragments do not
# carry the motif, because the cleavage falls inside it or outside it.
seen_peptides = set()
total_peptides = 0
stats_per_proteome = defaultdict(int)
stats_per_enzyme = defaultdict(int)

# If the runtime restarts, the digestion resumes from the CSV already on disk.
# Writing goes to a .partial file that is renamed only when the loop reaches
# the end, so a file truncated by an interruption is never taken for a complete
# pool.
PEPTIDES_PARTIAL = PEPTIDES_CSV + ".partial"
# Resuming is only valid for a file renamed at the end of the loop.
RESUME_HYDROLYSIS = (os.path.exists(PEPTIDES_CSV)
              and os.path.getsize(PEPTIDES_CSV) > 1000)
if RESUME_HYDROLYSIS:
    print(f"{os.path.basename(PEPTIDES_CSV)} already present: "
          f"hydrolysis not re-run\n")
else:
    if os.path.exists(PEPTIDES_PARTIAL):
        print(f"{os.path.basename(PEPTIDES_PARTIAL)}: a previous digestion was "
              f"interrupted, starting over\n")
    print("six proteases, Keil proline rule, fragments of "
          f"{MIN_LEN}-{MAX_LEN} residues")
    print(f"proteins from STEP 3, and only fragments carrying {MOTIF}\n")

with open(os.devnull if RESUME_HYDROLYSIS else PEPTIDES_PARTIAL, "w", newline="") as f_out:
    writer = csv.writer(f_out)
    if not RESUME_HYDROLYSIS:
        writer.writerow(["peptide", "length", "protein_id", "protein_name",
                         "source", "position", "enzyme"])

    for pname, pdata in ({} if RESUME_HYDROLYSIS else active_proteomes).items():
        fpath = motif_fasta.get(pname)          # STEP 3 decides which file
        if not fpath or not os.path.exists(fpath):
            continue

        n_seq = motif_proteomes.get(pname, 0)
        pep_count = 0

        for header, seq in tqdm(parse_fasta(fpath), total=n_seq,
                                desc=f"{pdata['name'][:20]:20s}"):
            clean = "".join(c for c in seq if c in VALID_AA)
            if len(clean) < MIN_LEN:
                continue

            prot_id = header.split("|")[1] if "|" in header else header.split()[0]
            prot_name = header.split("|")[2].split(" OS=")[0] if "|" in header and "OS=" in header else header[:80]

            for enz_name, enz_cuts in ENZYMES.items():
                for frag, start_pos in enzymatic_digest(clean, enz_cuts):
                    # One row per occurrence, aggregated at STEP 5. A global dedup here
                    # would credit each peptide to the first proteome and the first enzyme
                    # met, losing the other origins without leaving a trace.
                    if not RECORD_ALL_OCCURRENCES and frag in seen_peptides:
                        continue
                    if not _motif_rx.search(frag):
                        continue      # second level: only motif-bearing fragments
                    if not passes_biofilter(frag):
                        continue
                    # With RECORD_ALL_OCCURRENCES True the set is never read, so it is not
                    # filled: it would be some 6 million strings, 0.6 GB held for the rest
                    # of the session.
                    if not RECORD_ALL_OCCURRENCES:
                        seen_peptides.add(frag)
                    writer.writerow([frag, len(frag), prot_id, prot_name,
                                    pname, f"{start_pos+1}-{start_pos+len(frag)}",
                                    enz_name])
                    total_peptides += 1
                    pep_count += 1
                    stats_per_enzyme[enz_name] += 1

        stats_per_proteome[pname] = pep_count

# Renaming here is the only evidence that the digestion reached the end.
if not RESUME_HYDROLYSIS:
    os.replace(PEPTIDES_PARTIAL, PEPTIDES_CSV)

# On a resume the counters are rebuilt by reading the file, so the cells
# downstream get the same numbers as a complete run.
if RESUME_HYDROLYSIS:
    with open(PEPTIDES_CSV, newline="") as _f:
        _r = csv.reader(_f)
        next(_r, None)
        for _row in _r:
            if len(_row) < 7:
                continue
            total_peptides += 1
            stats_per_proteome[_row[4]] += 1
            stats_per_enzyme[_row[6]] += 1

print(f"\n{'='*64}")
print(f"   proteins digested             {sum(motif_proteomes.values()):>12,}"
      f"   (STEP 3 kept these)")
print(f"   occurrences with {MOTIF}      {total_peptides:>12,}")
print()
print("   This is the candidate pool. The library counts - every protein and")
print("   every fragment - are not produced here and are quoted in the Methods.")
print(f"\nper proteome:")
for pname, count in sorted(stats_per_proteome.items(), key=lambda x: -x[1]):
    print(f"   {active_proteomes[pname]['name']:25s}{count:>10,}")
print(f"\nper protease:")
for enz, count in sorted(stats_per_enzyme.items(), key=lambda x: -x[1]):
    print(f"   {enz:25s}{count:>10,}")
# Per-protease and per-proteome counts go to a file, not only to the screen:
# the Methods section needs them.
_counts_path = os.path.join(RESULTS_DIR, "a_proteomes_and_library",
                            "counts_by_protease_and_proteome.csv")
os.makedirs(os.path.dirname(_counts_path), exist_ok=True)
with open(_counts_path, "w", newline="") as _f:
    _w = csv.writer(_f)
    _w.writerow(["category", "name", "peptides"])
    for _e, _n in sorted(stats_per_enzyme.items(), key=lambda x: -x[1]):
        _w.writerow(["protease", _e, _n])
    for _p, _n in sorted(stats_per_proteome.items(), key=lambda x: -x[1]):
        _w.writerow(["proteome", _p, _n])
    _w.writerow(["total", "peptides", total_peptides])
print(f"\nwritten:\n   {PEPTIDES_CSV}\n   {_counts_path}")


## STEP 5a — Physicochemical pre-filter

Applied sequentially: Boman ≥ 1.0, GRAVY between -2.0 and +0.5, net charge
between -2 and +4, MW ≤ 1600 Da, hydrophobic fraction 30–60%.

**The instability index is not applied**: it rejects PMI (47.42) and p53(17-29)
(58.59), 2 of the 3 validated binders.

In [ ]:
# ============================================================
# STEP 5a - PHYSICOCHEMICAL PRE-FILTER
# ============================================================
import csv, os

print("physicochemical pre-filter\n")

# Radzicka & Wolfenden 1988: ΔG cyclohexane → water transfer (kcal/mol)
# Boman Index = -(sum ΔG) / n_residues
BOMAN_TRANSFER = {
    "A": 0.50, "R": -2.53, "N": -6.64, "D": -8.72, "C": -1.29,
    "Q": -5.54, "E": -6.81, "G": 1.15, "H": -4.66, "I": 4.92,
    "K": -5.55, "L": 4.92, "M": 2.35, "F": 2.98, "P": -0.02,
    "S": -3.40, "T": -2.57, "W": 2.33, "Y": -0.14, "V": 4.04,
}
CHARGE_AA = {"R": 1, "K": 1, "H": 0.5, "D": -1, "E": -1}
HYDRO_KD = {"A":1.8,"R":-4.5,"N":-3.5,"D":-3.5,"C":2.5,"Q":-3.5,"E":-3.5,"G":-0.4,
            "H":-3.2,"I":4.5,"L":3.8,"K":-3.9,"M":1.9,"F":2.8,"P":-1.6,"S":-0.8,
            "T":-0.7,"W":-0.9,"Y":-1.3,"V":4.2}
MW_AA = {"A":89.1,"R":174.2,"N":132.1,"D":133.1,"C":121.2,"Q":146.2,"E":147.1,"G":75.0,
         "H":155.2,"I":131.2,"L":131.2,"K":146.2,"M":149.2,"F":165.2,"P":115.1,"S":105.1,
         "T":119.1,"W":204.2,"Y":181.2,"V":117.1}

def boman_index(seq):
    """Boman 2003: protein-protein interaction index. Higher = more likely binder."""
    total = sum(BOMAN_TRANSFER.get(aa, 0) for aa in seq)
    return -total / len(seq) if seq else 0

def net_charge(seq):
    return sum(CHARGE_AA.get(aa, 0) for aa in seq)

def gravy(seq):
    vals = [HYDRO_KD.get(aa, 0) for aa in seq]
    return sum(vals) / len(vals) if vals else 0

def mol_weight(seq):
    return sum(MW_AA.get(aa, 0) for aa in seq) - (len(seq) - 1) * 18.02

def hydrophobic_ratio(seq):
    HYDROPHOBIC = set("AILMFPWV")
    return sum(1 for aa in seq if aa in HYDROPHOBIC) / len(seq) if seq else 0

target_site_charges = {}
for tname, tdata in active_targets.items():
    site_res = tdata.get("binding_site_residues", "")
    site_charge = net_charge(site_res)
    target_site_charges[tname] = site_charge
    print(f"   {tdata['name']}: site charge = {site_charge:+.1f}")

# Reading and writing are streamed one row at a time, at constant memory: held
# in memory the file would be several GB on a 12.7 GB runtime.
print(f"\nreading peptides from {PEPTIDES_CSV}")
FILTERED_CSV = os.path.join(RESULTS_DIR, "peptides_prefiltered.csv")
FILTERED_PARTIAL = FILTERED_CSV + ".partial"


total_before = 0

# ── FINAL THRESHOLDS ──
MIN_BOMAN = 1.0          # Boman 2003, validated: DVKECKLSKIF=1.59 passes
MIN_GRAVY = -2.0         # Kyte-Doolittle
MAX_GRAVY = 0.5          # Prevents aggregation
MIN_CHARGE = -2           # Wide range
MAX_CHARGE = 4            # Wide range
MAX_MW = 1600             # Bioavailability
MIN_HYDRO = 0.30          # PPI interfaces
MAX_HYDRO = 0.60          # PPI interfaces

# Resuming is only valid for a file renamed at the end of the loop, never for
# one truncated by an interruption.
RESUME_PREFILTER = (os.path.exists(FILTERED_CSV)
              and os.path.getsize(FILTERED_CSV) > 1000)
if RESUME_PREFILTER:
    print(f"   {os.path.basename(FILTERED_CSV)} already present: "
          f"pre-filter not re-run")

stats = {"boman": 0, "gravy": 0, "charge": 0, "mw": 0, "hydro": 0, "passed": 0}
total_after = 0
_writer = None
_f_out = open(os.devnull if RESUME_PREFILTER else FILTERED_PARTIAL, 'w', newline='')
_f_in = open(PEPTIDES_CSV, 'r')

for pep in (iter(()) if RESUME_PREFILTER else csv.DictReader(_f_in)):
    total_before += 1
    seq = pep["peptide"]
    
    # 1. Boman ≥ 1.0
    if boman_index(seq) < MIN_BOMAN:
        stats["boman"] += 1
        continue
    
    # 2. GRAVY -2.0 to +0.5
    g = gravy(seq)
    if g < MIN_GRAVY or g > MAX_GRAVY:
        stats["gravy"] += 1
        continue
    
    # 3. Charge -2 to +4
    ch = net_charge(seq)
    if ch < MIN_CHARGE or ch > MAX_CHARGE:
        stats["charge"] += 1
        continue
    
    # 4. Molecular weight, against MAX_MW = 1600 Da
    if mol_weight(seq) > MAX_MW:
        stats["mw"] += 1
        continue
    
    # 5. Hydrophobicity 30-60%
    hr = hydrophobic_ratio(seq)
    if hr < MIN_HYDRO or hr > MAX_HYDRO:
        stats["hydro"] += 1
        continue
    
    if _writer is None:
        _writer = csv.DictWriter(_f_out, fieldnames=list(pep.keys()))
        _writer.writeheader()
    _writer.writerow(pep)
    total_after += 1
    stats["passed"] += 1

_f_in.close()
_f_out.close()
if not RESUME_PREFILTER:
    os.replace(FILTERED_PARTIAL, FILTERED_CSV)


def _count_rows(path):
    with open(path) as _f:
        return max(0, sum(1 for _ in _f) - 1)


if RESUME_PREFILTER:
    total_before = _count_rows(PEPTIDES_CSV)
    total_after = _count_rows(FILTERED_CSV)
    stats["passed"] = total_after

PEPTIDES_CSV = FILTERED_CSV
pct = (1 - total_after / total_before) * 100 if total_before > 0 else 0

print(f"\n{'='*50}")
print(f"   Before:  {total_before:>12,}")
print(f"   After:   {total_after:>12,}")
print(f"   Removed: {total_before - total_after:>11,} ({pct:.1f}%)")
print(f"{'='*50}")
if RESUME_PREFILTER:
    print("   (resumed: the per-criterion breakdown is not available, "
          "the filter was not re-run)")
print(f"   ❌ Boman < 1.0:       {stats['boman']:>10,}")
print(f"   ❌ GRAVY out of range:{stats['gravy']:>10,}")
print(f"   ❌ Charge out -2/+4:  {stats['charge']:>10,}")
print(f"   ❌ MW > {MAX_MW}:         {stats['mw']:>10,}")
print(f"   ❌ Hydrophob. out:    {stats['hydro']:>10,}")
print(f"   ✅ Passed:            {stats['passed']:>10,}")
print(f"{'='*50}")
print(f"\n✅ ESM-2 will run on {total_after:,} peptides")


## STEP 5b — Aggregation: from occurrences to distinct peptides

STEP 4 wrote **one row per occurrence**: the same peptide can be released from
different proteins, in different species, by different proteases. The
per-protease and per-species counts are taken here, after aggregation.

This stage reads what STEP 5a left, `peptides_prefiltered.csv`, so it aggregates
the occurrences that passed the pre-filter. Rows are grouped by sequence, and
every species and every protease that generates a peptide is kept, in the `sources` and `enzymes` columns. What comes
out is the **candidate set**, one row per distinct peptide — 3,214 of them — and
it is what ESM-2 orders at STEP 6. Both counts are reported, occurrences and
distinct peptides.

The motif is applied here a third time, as a check: STEPs 3 and 4 already kept
only what carries it, so a peptide removed at this point would mean the stages
ran out of order.

In [ ]:
# ============================================================
# STEP 5b - AGGREGATION TO DISTINCT PEPTIDES
# One row per distinct peptide, with all of its origins.
# ============================================================
import pandas as pd
import os

# The aggregation STEP 4 deferred: one row per distinct peptide, keeping the
# multi-species and multi-protease attribution in the sources/enzymes columns.
df_peptides = pd.read_csv(PEPTIDES_CSV)
_occurrences = len(df_peptides)
df_peptides = df_peptides.groupby("peptide", as_index=False).agg(
    length=("length", "first"),
    protein_id=("protein_id", "first"),
    protein_name=("protein_name", "first"),
    source=("source", "first"),
    sources=("source", lambda s: ";".join(sorted(set(s)))),
    n_sources=("source", "nunique"),
    position=("position", "first"),
    enzyme=("enzyme", lambda s: ";".join(sorted(set(s)))),
    n_enzymes=("enzyme", "nunique"),
)
print(f"   {_occurrences:,} occurrences -> {len(df_peptides):,} distinct peptides")

# ── STAGE ORDER ─────────────────────────────────
# The motif is applied before the ranking, so the candidate set is complete
# when ESM-2 sees it and ESM-2 only orders it.
PHARMACOPHORE_REGEX = MOTIF      # the motif declared once, at STEP 3

# Applied a third time, as a check: STEPs 3 and 4 already kept only the
# proteins and the fragments that carry it, so here there is nothing left to
# remove, and a peptide removed at this point would mean the stages ran out of
# order.
_before = len(df_peptides)
df_peptides = df_peptides[
    df_peptides["peptide"].str.contains(PHARMACOPHORE_REGEX, regex=True)
].reset_index(drop=True)
if len(df_peptides) == _before:
    print(f"   pharmacophore {PHARMACOPHORE_REGEX}: nothing to remove, the pool "
          f"already carries the motif only")
else:
    print(f"   ANOMALY: {_before - len(df_peptides):,} peptides removed here. "
          f"STEPs 3 and 4 should have left none.")

print(f"   {len(df_peptides):,} peptides enter ESM-2\n")



## STEP 6 — ESM-2 similarity ranking

Per-residue representations are pooled with a weighted mean in which the first
position receives weight 0.3, to attenuate the initiator-methionine bias;
special tokens are excluded. Cosine similarity is computed against the reference
peptide.

Nothing is discarded here and there is no cut-off: ESM-2 supplies the order in
which the candidate set is submitted for co-folding. The candidate of the
manuscript is 17th of 3,214.

TF32 is disabled explicitly: on Ampere GPUs it runs the matrix products at about
1e-3 relative precision, while the step between adjacent ranks is of order
1e-6.

In [ ]:
from datetime import datetime
# ============================================================
# STEP 6 — ESM-2 SIMILARITY RANKING
# ============================================================
import torch
from transformers import AutoTokenizer, AutoModel
import pandas as pd
from tqdm.auto import tqdm
import heapq
import gc
import os
import time
import json

# This cell inherits PEPTIDES_CSV, WORK_DIR, RESULTS_DIR, SELECTED_DBS and
# active_targets from the cells above. A runtime restart clears them, and the
# check below reports that rather than letting a bare NameError escape.
try:
    PEPTIDES_CSV, WORK_DIR, RESULTS_DIR, SELECTED_DBS, active_targets
except NameError as _e:
    raise RuntimeError(
        "The runtime was restarted: the state of the previous cells is lost "
        "(" + str(_e) + ").\n"
        "Re-run from the top with Runtime > Run before. STEPs 4 and 5a "
        "recognise what they already produced\nand do not repeat the digestion, "
        "so it takes seconds to get back here."
    ) from None

# PEPTIDES_CSV points at the raw hydrolysis file until STEP 5a reassigns it.
# Without this check, with STEP 5a not run, this stage would rank the
# unfiltered pool, four times larger: no exception, just twenty hours of
# scanning.
if not str(PEPTIDES_CSV).endswith("peptides_prefiltered.csv"):
    raise RuntimeError(
        "PEPTIDES_CSV still points at the raw hydrolysis file\n("
        + str(PEPTIDES_CSV) + ").\nRun STEP 5a before this cell: without the "
        "pre-filter the pool is about four times\nlarger and the scan does not "
        "finish."
    )

# ── ESM-2 Setup ──
# On Ampere and later GPUs the matrix products can run in TF32, which has a
# 10-bit mantissa: a relative precision near 1e-3. On this ranking, packed as
# it is, that is worth hundreds of ranks. It has to be turned off explicitly,
# or the result depends on which GPU the runtime was assigned.
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

device = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "facebook/esm2_t33_650M_UR50D"
print(f"loading ESM-2 ({MODEL_NAME}) on {device}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, clean_up_tokenization_spaces=False)
model = AutoModel.from_pretrained(MODEL_NAME, add_pooling_layer=False).to(device).eval()
# NOTE: the model stays in float32. Between adjacent ranks the similarity
# step is of order 1e-6, while fp16 carries a relative precision near 0.99 of
# about 1e-3 - hundreds of ranks of error.
if torch.cuda.device_count() > 1:
    model = torch.nn.DataParallel(model)
    print(f"using {torch.cuda.device_count()} GPUs")
print("model loaded\n")

# ── Embedding function with reduced weight on initial M ──
def get_embedding(sequence, reduce_first=True):
    """Compute ESM-2 embedding with reduced weight on the first position."""
    inputs = tokenizer(sequence, return_tensors="pt", add_special_tokens=True, max_length=1024, truncation=True).to(device)
    with torch.no_grad():
        out = model(**inputs)
    positions = out.last_hidden_state[0, 1:-1, :]  # Exclude [CLS] and [EOS]
    if reduce_first and len(positions) > 1:
        weights = torch.ones(positions.shape[0], device=device)
        weights[0] = 0.3  # First position weighted less (initial M bias)
        weights = weights / weights.sum()
        emb = (positions * weights.unsqueeze(1)).sum(dim=0)
    else:
        emb = positions.mean(dim=0)
    del inputs, out
    return emb

print("loading the candidate set")

# ── Mining per target ──
CHUNK_SIZE = 256 * max(1, torch.cuda.device_count())
# No truncation: TOP_K is the size of the candidate set, so the whole set is
# ordered and none of it is discarded.
TOP_K = len(df_peptides)
print(f"   no truncation: all {TOP_K:,} candidates are ordered")
CHECKPOINT_FILE = os.path.join(RESULTS_DIR, "esm2_checkpoint.json")
_db_suffix = "_".join(SELECTED_DBS)
_date_str = datetime.now().strftime("%Y%m%d")

completed_targets = set()
if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE) as f:
        checkpoint = json.load(f)
        completed_targets = set(checkpoint.get("completed", []))
    print(f"checkpoint found: {len(completed_targets)} targets already complete")

all_target_results = {}

for target_name, target_data in sorted(active_targets.items(), key=lambda x: x[1]["priority"]):
    if target_name in completed_targets:
        results_file = os.path.join(RESULTS_DIR, "b_candidates",
            f"esm2_top{TOP_K}_{target_name}_{_db_suffix}_{_date_str}.json")
        os.makedirs(os.path.dirname(results_file), exist_ok=True)
        if os.path.exists(results_file):
            with open(results_file) as f:
                all_target_results[target_name] = json.load(f)
            print(f"⏭️ {target_data['name']} — already complete, loaded {len(all_target_results[target_name])} results")
            continue

    print(f"\n{'='*60}")
    print(f"target: {target_data['name']}")
    print(f"   reference peptide: {target_data['ref_peptide']}")
    print(f"{'='*60}")

    ref_pep = target_data["ref_peptide"] or target_data["binding_site_residues"]
    ref_emb = get_embedding(ref_pep)
    print(f"   Using: {ref_pep}")

    # The heap is saved every SAVE_EVERY batches, so an interrupted scan resumes
    # from the last save and not from the beginning.
    ESM_PARTIAL = os.path.join(RESULTS_DIR, f"esm2_partial_{target_name}.json")
    SAVE_EVERY = 200
    top_k_heap = []
    _resume_from = 0
    if os.path.exists(ESM_PARTIAL):
        with open(ESM_PARTIAL) as _f:
            _p = json.load(_f)
        _resume_from = _p["next_index"]
        top_k_heap = [(_d["esm2_similarity"], _k, _d)
                      for _k, _d in enumerate(_p["heap"])]
        heapq.heapify(top_k_heap)
        print(f"   resumed from peptide {_resume_from:,} "
              f"({len(top_k_heap):,} already in the queue)")
    t0 = time.time()

    for i in tqdm(range(_resume_from, len(df_peptides), CHUNK_SIZE),
                  desc=f"ESM-2 {target_data['name'][:20]}"):
        chunk = df_peptides.iloc[i:i+CHUNK_SIZE]
        batch_seqs = chunk["peptide"].tolist()

        inputs = tokenizer(batch_seqs, return_tensors="pt", padding=True,
                          truncation=True, max_length=1024,
                          add_special_tokens=True).to(device)
        with torch.no_grad():
            out = model(**inputs)

        mask = inputs["attention_mask"]

        for j, row in enumerate(chunk.itertuples()):
            seq_len = mask[j].sum().item()
            if seq_len > 2:
                emb_positions = out.last_hidden_state[j, 1:seq_len-1, :]
                w = torch.ones(emb_positions.shape[0], device=device)
                w[0] = 0.3
                w = w / w.sum()
                emb = (emb_positions * w.unsqueeze(1)).sum(dim=0)
            else:
                emb = out.last_hidden_state[j, 0, :]

            sim = torch.nn.functional.cosine_similarity(
                ref_emb.unsqueeze(0), emb.unsqueeze(0)
            ).item()

            pep_dict = {
                "peptide": row.peptide,
                "length": row.length,
                "protein_id": row.protein_id,
                "protein_name": row.protein_name,
                "source": row.source,
                "position": row.position,
                "enzyme": row.enzyme,
                # The full species and protease attribution reaches the results
                # file: every entry carries all of its origins.
                "sources": row.sources,
                "n_sources": int(row.n_sources),
                "n_enzymes": int(row.n_enzymes),
                "esm2_similarity": round(sim, 6),
                "target": target_name,
            }

            if len(top_k_heap) < TOP_K:
                heapq.heappush(top_k_heap, (sim, id(pep_dict), pep_dict))
            else:
                heapq.heappushpop(top_k_heap, (sim, id(pep_dict), pep_dict))

        del inputs, out, mask
        torch.cuda.empty_cache()

        if (i // CHUNK_SIZE) % SAVE_EVERY == 0 and i > _resume_from:
            with open(ESM_PARTIAL, "w") as _f:
                json.dump({"next_index": i + CHUNK_SIZE,
                           "heap": [_it[2] for _it in top_k_heap]}, _f)

    elapsed = time.time() - t0
    results = [item[2] for item in sorted(top_k_heap, key=lambda x: x[0], reverse=True)]
    all_target_results[target_name] = results

    results_file = os.path.join(
        RESULTS_DIR, "b_candidates",
        f"esm2_top{TOP_K}_{target_name}_{_db_suffix}_{_date_str}.json")
    os.makedirs(os.path.dirname(results_file), exist_ok=True)
    with open(results_file, "w") as f:
        json.dump(results, f, indent=2)

    if os.path.exists(ESM_PARTIAL):
        os.remove(ESM_PARTIAL)
    completed_targets.add(target_name)
    with open(CHECKPOINT_FILE, "w") as f:
        json.dump({"completed": list(completed_targets)}, f)

    print(f"\n✅ {target_data['name']} done in {elapsed:.1f}s")
    print(f"   Speed: {len(df_peptides)/elapsed:.0f} peptides/sec")
    # The first thirty: the ones STEP 7 sends to co-folding.
    _N_SHOW = 30
    _lead = "SPAFESTWDILK"
    print(f"   first {_N_SHOW} by similarity:")
    for k, r in enumerate(results[:_N_SHOW], 1):
        _m = "  <-- candidate of the manuscript" if r["peptide"] == _lead else ""
        print(f"   #{k:<3}{r['peptide']:16s}{len(r['peptide']):>4}"
              f"  sim={r['esm2_similarity']:.6f}  {r['source']}{_m}")
    _pos = [k for k, r in enumerate(results, 1) if r["peptide"] == _lead]
    if _pos:
        print(f"\n   {_lead}: rank {_pos[0]} of {len(results)}")
    else:
        print(f"\n   {_lead} is not in this ranking")

print(f"\n{'='*60}")
print(f"✅ ESM-2 mining complete for {len(all_target_results)} targets!")
for tn, res in all_target_results.items():
    print(f"   {active_targets[tn]['name']}: {len(res)} candidates")


## STEP 7 — The first 30, and their AlphaFold 3 jobs

The candidate set is annotated with each peptide's motif match and spacing,
written out with its ranks, and the **first 30 by ESM-2 order** become an
AlphaFold 3 job list. STEP 10 measures how weak that ordering is as a filter.

Receptor and seed are those of every complex in this work.

Run the job list on the AlphaFold 3 Server and give the archive it returns
to the picker STEP 8 opens.

In [ ]:
# ============================================================
# STEP 7 - THE FIRST THIRTY, AND THEIR AF3 JOBS
# ============================================================
# The ordering comes from STEP 6 already sorted; the motif is the one declared
# at STEP 3. Nothing is re-filtered and nothing is re-sorted here.
import json, os, collections

TOP_N = 30
OUT_BATCH = os.path.join(OUT_ROOT, "d_top30_screening")
os.makedirs(OUT_BATCH, exist_ok=True)

_rx = _re_module.compile(MOTIF)

candidates = []
for e in all_target_results[TARGET]:
    m = _rx.search(e["peptide"])
    if not m:
        # Every candidate carries the motif by now; a peptide without it would
        # mean the stages ran out of order.
        print(f"   ANOMALY: {e['peptide']} does not carry {MOTIF}")
        continue
    e = dict(e)
    e["pharmacophore"] = MOTIF
    e["pharmacophore_match"] = m.group()
    e["spacing"] = "i+" + str(len(m.group()) - 1)
    candidates.append(e)
for _r, _e in enumerate(candidates, 1):
    _e["rank_in_set"] = _r

_n_species = len({e.get("source") for e in candidates})
print("candidate set: %d peptides from %d species, ordered by ESM-2 similarity"
      % (len(candidates), _n_species))

_cand_dir = os.path.join(RESULTS_DIR, "b_candidates")
os.makedirs(_cand_dir, exist_ok=True)
_set_file = os.path.join(
    _cand_dir, f"candidate_set_{TARGET}_{_db_suffix}_{_date_str}.json")
with open(_set_file, "w") as f:
    json.dump(candidates, f, indent=2)

# ── the first thirty ────────────────────────────────────────────────────────
top = candidates[:TOP_N]
clash = [e["peptide"] for e in top if e["peptide"] in REFERENCE_SEQUENCES]
if clash:
    print("   ANOMALY: these candidates are reference peptides and cannot be")
    print("   screened against their own class: %s" % ", ".join(clash))

jobs, manifest = [], []
for i, e in enumerate(top, 1):
    pep = e["peptide"]
    jobs.append(af3_job("%s_TOP%d_R%02d_%s" % (TARGET, TOP_N, i, pep), pep))
    manifest.append({"esm_rank": i, "peptide": pep, "length": len(pep),
                     "class": "candidate (no label)",
                     "esm2_similarity": e.get("esm2_similarity"),
                     "spacing": e.get("spacing"),
                     "motif_match": e.get("pharmacophore_match"),
                     "species": e.get("source"), "protein_id": e.get("protein_id"),
                     "position": e.get("position"), "enzyme": e.get("enzyme")})

p_jobs = os.path.join(OUT_BATCH, "af3_jobs_canonical_top30.json")
p_man = os.path.join(OUT_BATCH, "af3_batch_manifest.csv")
with open(p_jobs, "w") as f:
    json.dump(jobs, f, indent=1)
with open(p_man, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(manifest[0].keys()))
    w.writeheader(); w.writerows(manifest)

print("\n%3s  %-16s%4s%11s  %-5s%s" % ("#", "peptide", "len", "ESM", "sp", "species"))
for r in manifest:
    print("%3d  %-16s%4d%11.6f  %-5s%s"
          % (r["esm_rank"], r["peptide"], r["length"], r["esm2_similarity"],
             str(r["spacing"] or "-"), r["species"]))

print("\nwritten:\n   %s\n   %s\n   %s" % (_set_file, p_jobs, p_man))
print("\nNEXT: run these %d jobs on AlphaFold 3 Server and place the archive in"
      % len(jobs))
print("'%s'. STEP 8 reads it." % EXTERNAL)

# Whether the thirty are already in an archive decides the wording only: the
# job list is handed over either way.
_pool, _ = complexes_by_sequence(verbose=False)
_to_run = [m["peptide"] for m in manifest if m["peptide"] not in _pool]
if _to_run:
    print("\n%d of the 30 have not been co-folded yet." % len(_to_run))
# The job list is the file that leaves the notebook; the manifest stays in the
# run directory.
deliver(p_jobs, needed=bool(_to_run), n_jobs=len(jobs))
print("   the manifest of the 30 stays in the run directory:")
print("   %s" % p_man)

## STEP 8 — Screening, and the structures

Reads the archive without extracting it, takes the `summary_confidences` files
of every job — 5 per job, as the server returns them — and applies the threshold
measured on the calibration panel.

Before anything is reported the stage checks that every complex is on the same
85-residue construct, and reports as an anomaly any job whose receptor differs.

The hit rate comes with a Wilson 95% interval, and it is the rate on these 30
peptides, not the rate of the screen as a whole.

The stage then writes one two-chain PDB per peptide above the threshold — chain
A the receptor renumbered from 1, chain B the peptide renumbered from 1, heavy
atoms, pLDDT in the B-factor column — and the index groups a dynamics run
needs.

In [ ]:
# ============================================================
# STEP 8 - SCREENING AND STRUCTURES
# ============================================================
OUT_SCREEN = os.path.join(OUT_ROOT, "d_top30_screening")   # same as the jobs
OUT_PDB = os.path.join(OUT_ROOT, "i_structures")
os.makedirs(OUT_SCREEN, exist_ok=True)
os.makedirs(OUT_PDB, exist_ok=True)

# THRESHOLD comes from STEP 1, which measured it on the calibration panel. This
# is the stage that applies it, so this is where its absence stops the run.
if CALIBRATION is None:
    raise RuntimeError(
        "STOP - the threshold %.2f has not been validated on this target.\n\n"
        "STEP 1 writes the %d jobs of the calibration panel. Run them on the\n"
        "AlphaFold 3 Server, re-run STEP 1 and choose the archive it asks for,\n"
        "then come back here. Six jobs, and they are what the threshold rests\n"
        "on." % (THRESHOLD, 6))

_wanted = [m["peptide"] for m in manifest]     # the thirty of STEP 7
print("reading the thirty back:")
_got, _twice = require(_wanted, "the screening of the first thirty",
                         "af3_results_canonical_top30.zip")

screened, passed = [], []
if _got:
    for c in _got.values():
        m = st.mean(c["iptm"])
        screened.append({"peptide": c["peptide"], "length": len(c["peptide"]),
                         "n_models": len(c["iptm"]), "iptm_mean": round(m, 4),
                         "iptm_sd": round(st.stdev(c["iptm"]), 4) if len(c["iptm"]) > 1 else 0.0,
                         "iptm_best": round(max(c["iptm"]), 4),
                         "receptor_len": c["receptor_len"],
                         "archive": c["archive"], "job": c["job"],
                         "above_threshold": "yes" if m >= THRESHOLD else "no"})

if screened:
    lens = {r["receptor_len"] for r in screened}
    if lens != {len(RECEPTOR)}:
        print("   ANOMALY: receptors of different length in one batch: %s" % sorted(lens))
        print("   A screen run on more than one construct is not one screen.")
    ref = [r["peptide"] for r in screened if r["peptide"] in REFERENCE_SEQUENCES]
    if ref:
        print("   ANOMALY: reference peptides inside the screened batch: %s" % ", ".join(ref))

    # Sorted by mean so the table can be read, with no position number: above
    # the threshold the peptides are not separable, and a numbered column reads
    # as a ranking they do not support.
    screened.sort(key=lambda r: -r["iptm_mean"])
    print("\n%-16s%8s%8s%7s%5s  >= %.2f"
          % ("peptide", "mean", "SD", "best", "n", THRESHOLD))
    for r in screened:
        print("%-16s%8.3f%8.4f%7.2f%5d  %s"
              % (r["peptide"], r["iptm_mean"], r["iptm_sd"], r["iptm_best"],
                 r["n_models"], r["above_threshold"]))

    passed = [r for r in screened if r["above_threshold"] == "yes"]
    n, k = len(screened), len(passed)
    ph = k / n
    z = 1.96
    den = 1 + z * z / n
    centre = (ph + z * z / (2 * n)) / den
    half = z * math.sqrt(ph * (1 - ph) / n + z * z / (4 * n * n)) / den
    print("\nhit rate  %d of %d = %.0f%%   Wilson 95%% interval %.0f%% - %.0f%%"
          % (k, n, 100 * ph, 100 * (centre - half), 100 * (centre + half)))
    print("This is the rate on these %d peptides, not the rate of the screen as a whole." % n)
    t = os.path.join(OUT_SCREEN, "table_af3_screening.csv")
    with open(t, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(screened[0].keys()))
        w.writeheader(); w.writerows(screened)
    s = os.path.join(OUT_SCREEN, "af3_screening_summary.txt")
    with open(s, "w") as f:
        f.write("archives       %s\n" % ", ".join(sorted({r["archive"] for r in screened})))
        f.write("complexes      %d, receptor %s(25-109), %d residues\n"
                % (n, TARGET, len(RECEPTOR)))
        f.write("statistic      mean over the five models\n")
        f.write("threshold      iPTM >= %.2f%s\n"
                % (THRESHOLD, "" if CALIBRATION is None else ", measured on the calibration panel"))
        f.write("hit rate       %d/%d = %.0f%%   Wilson 95%% %.0f%%-%.0f%%\n"
                % (k, n, 100 * ph, 100 * (centre - half), 100 * (centre + half)))
    print("\nwritten:\n   %s\n   %s" % (t, s))


# ── the structures ──────────────────────────────────────────────────────────
def atoms_of(model_text):
    cols, inloop, out = [], False, []
    for line in model_text.split("\n"):
        s = line.strip()
        if s.startswith("_atom_site."):
            cols.append(s.split(".", 1)[1]); inloop = True; continue
        if inloop and cols:
            if not s or s.startswith("#") or s.startswith("loop_") or s.startswith("_"):
                if out:
                    break
                continue
            f = s.split()
            if len(f) < len(cols):
                continue
            d = dict(zip(cols, f))
            if d.get("group_PDB", "ATOM") != "ATOM" or d.get("type_symbol") == "H":
                continue
            out.append((d.get("label_asym_id"), int(d["label_seq_id"]), d.get("label_comp_id"),
                        d.get("label_atom_id"), float(d["Cartn_x"]), float(d["Cartn_y"]),
                        float(d["Cartn_z"]), float(d.get("B_iso_or_equiv", 0.0)),
                        d.get("type_symbol", "")))
    return out


def write_pdb(atoms, dest):
    order = []
    for a in atoms:
        if a[0] not in order:
            order.append(a[0])
    if len(order) != 2:
        raise RuntimeError("the model has %d chains, two are required" % len(order))
    count = {c: len({a[1] for a in atoms if a[0] == c}) for c in order}
    order.sort(key=lambda c: -count[c])            # the longer chain is the receptor
    tag = {order[0]: "A", order[1]: "B"}
    lines, serial = [], 0
    for asym in order:
        first = None
        for a in atoms:
            if a[0] != asym:
                continue
            if first is None:
                first = a[1]
            serial += 1
            name = a[3] if len(a[3]) >= 4 else " " + a[3].ljust(3)
            lines.append("ATOM  %5d %-4s%1s%3s %1s%4d%1s   %8.3f%8.3f%8.3f%6.2f%6.2f          %2s  "
                         % (serial, name, " ", a[2], tag[asym], a[1] - first + 1, " ",
                            a[4], a[5], a[6], 1.00, a[7], a[8].rjust(2)))
        lines.append("TER")
    lines.append("END")
    with open(dest, "w", newline="\n") as f:
        f.write("\n".join(lines) + "\n")
    return count[order[0]], count[order[1]], sum(1 for l in lines if l.startswith("ATOM"))


if passed:
    # Each structure is read from the archive that carried it: several archives
    # can sit in the folder at once.
    want = {r["peptide"]: r for r in passed}
    made = []
    print("\n%-16s%5s%5s%7s  file" % ("peptide", "recA", "pepB", "atoms"))
    for seq in want:
        c = _got[seq]
        at = atoms_of(zipfile.ZipFile(c["zip"]).read(c["model"]).decode("utf8", "replace"))
        dest = os.path.join(OUT_PDB, "%s_1-%d_PEP_%d-%d_%s.pdb"
                            % (TARGET, len(RECEPTOR), len(RECEPTOR) + 1,
                               len(RECEPTOR) + len(seq), seq))
        nrec, npep, nat = write_pdb(at, dest)
        ok = (nrec == len(RECEPTOR) and npep == len(seq))
        made.append((seq, nrec, npep, ok))
        print("%-16s%5d%5d%7d  %s%s" % (seq, nrec, npep, nat, os.path.basename(dest),
                                        "" if ok else "   <-- CHECK"))

    idx = os.path.join(OUT_PDB, "md_index_groups.txt")
    with open(idx, "w") as f:
        f.write("# receptor and peptide index groups after 'pdb2gmx -merge all'\n")
        for seq, nrec, npep, _ in sorted(made, key=lambda t: -want[t[0]]["iptm_mean"]):
            f.write("%-16s iPTM %.3f   receptor 1-%-4d peptide %d-%d\n"
                    % (seq, want[seq]["iptm_mean"], nrec, nrec + 1, nrec + npep))
    print("\n%d structures in %s" % (len(made), OUT_PDB))
    print("index groups: %s" % idx)
    print("\nThe dynamics themselves are outside this notebook: they need a GPU and hours")
    print("of wall clock. Run them elsewhere and bring the results back to STEP 9.")

## STEP 9 — The candidate, and whether its mechanism is p53's

This stage needs one candidate, and the notebook does not choose it. The choice
is made outside, on the GROMACS dynamics of the peptides above the threshold:
5 ns for all 16, then 100 ns for 3 of them, in CHARMM36m at 310 K. What is read
from those runs is the backbone RMSD of the peptide, the inter-chain hydrogen
bonds, the minimum distance, the number of contacts and the radius of gyration.
The peptide that comes back is written into `CANDIDATE` below, with who decided
and on what basis.

49 jobs, all on the same 85-residue construct, 5 models, same seed:

| block | jobs | where the sequences come from |
|---|---|---|
| perturbation panel | 29 | written out in the cell |
| alanine scan of the candidate | 11 | every position in turn |
| p53 mirror | 3 | written out in the cell |
| the two combinations | 2 | written out in the cell |
| geometry: the tryptophan moved by one | 2 | a swap with the neighbour |
| references: the candidate and p53(17-29) unmutated | 2 | the two sequences themselves |

The 29 perturbations are the tryptophan moved along the sequence, anagrams with
the tryptophan replaced by alanine, hydrophobic substitutions, and peptides
drawn at random from the proteomes. They measure that the score falls on
deliberately perturbed sequences. They do not measure binders against
non-binders: 29 designed decoys are not experimentally validated negatives, and
that comparison is STEP 1.

The two combinations are the double at the two pocket residues and the triple
that adds the tryptophan.

The mirror on p53(17-29) is 3 variants, each paired with the corresponding
position of the candidate: Phe19 with the candidate's Phe, Trp23 with its Trp,
Leu25+Leu26 with the two pocket residues. The measurement is the pair: if the
two peptides lose confidence at the same positions and keep it at the same
positions, the mechanism is the same one.

In [ ]:
# ============================================================
# STEP 9 - CANDIDATE PANEL AND THE p53 MIRROR
# ============================================================
OUT_CAND = os.path.join(OUT_ROOT, "e_candidate_mechanism")
os.makedirs(OUT_CAND, exist_ok=True)

# ── the decision this notebook does not make ─────────────────────────────────
CANDIDATE            = ""    # the peptide carried forward
CANDIDATE_DECIDED_BY = ""    # who decided
CANDIDATE_BASIS      = ""    # on what basis, in one line

P53 = "ETFSDLWKLLPEN"        # p53(17-29), the peptide of the 1YCR co-crystal

# The 29 are perturbations of this peptide and of no other.
PERTURBATIONS_OF = "SPAFESTWDILK"
PERTURBATIONS = [
    ("WPAFESTDILK",   "W moved, position 1"),
    ("SWAFESTDILK",   "W moved, position 2"),
    ("SPWFESTDILK",   "W moved, position 3"),
    ("SPAFWSTDILK",   "W moved, position 5"),
    ("KLIDWTSEFAPS",  "W moved, sequence reversed"),
    ("SPAFESTDIWK",   "W moved, position 10"),
    ("SPAFESTDILWK",  "W moved, position 11"),
    ("SPAFESTDIKLW",  "W moved, C-terminus"),
    ("SPAFESTDILK",   "W to hydrophobic: deleted"),
    ("LPAFESTDILK",   "W to hydrophobic: S1L, no W"),
    ("SPAFESTVDILK",  "W to hydrophobic: W to V"),
    ("KPAFESTDILS",   "W to hydrophobic: ends swapped, no W"),
    ("SPAFESTGILK",   "W to hydrophobic: W to G"),
    ("TPAFESDFILK",   "W to hydrophobic: W to F, S1T"),
    ("APSFETDILKV",   "W to hydrophobic: reordered, no W"),
    ("FPASETDILKV",   "W to hydrophobic: reordered, no W"),
    ("VGVNGFGRIGR",   "random from the proteomes"),
    ("AGFAGDDAPR",    "random from the proteomes"),
    ("INVYYNEATGGK",  "random from the proteomes"),
    ("AILVDLEPGTMDS", "random from the proteomes"),
    ("DLYANTVLSGGTTM", "random from the proteomes"),
    ("VIHDHSGKVVK",   "random from the proteomes"),
    ("EIAQDFKTDLR",   "random from the proteomes"),
    ("LNVQNKDALLEK",  "random from the proteomes"),
    ("TPASSLFIADEK",  "anagram, W to A"),
    ("STASLIEDPAKF",  "anagram, W to A"),
    ("PKILASEDSATF",  "anagram, W to A"),
    ("DSLSFIPETAKA",  "anagram, W to A"),
    ("IPLATDSAESFK",  "anagram, W to A"),
]

# Anchors of the candidate, and the combinations: structural choices.
ANCHORS      = {4: "Phe19", 8: "Trp23", 10: "Leu26 pocket", 11: "Leu26 pocket"}
COMBINATIONS = [((10, 11), "the two pocket residues"),
                ((8, 10, 11), "the two pocket residues and the tryptophan")]

# The mirror: p53 position -> the candidate position it corresponds to.
# The Leu26 pocket is mirrored as the pair L25A+L26A against I10A+L11A, which
# is the substitution the two peptides were compared on; the single variants
# L25A, L26A, D21A and K24A were prepared and never co-folded.
MIRROR = [(19, 4,  "Phe19 anchor"),
          (23, 8,  "Trp23 anchor")]
MIRROR_PAIRS = [((25, 26), (10, 11), "the Leu26 pocket, both residues")]
P53_FIRST = 17               # p53(17-29): residue 17 is the first of the peptide


def substitute(peptide, positions, aa="A"):
    s = list(peptide)
    for p in positions:
        s[p - 1] = aa
    return "".join(s)


if not CANDIDATE:
    # Asked at run time when the three constants above are left empty.
    print("Which peptide goes forward? The choice is made outside, on the")
    print("GROMACS dynamics of the peptides above the threshold.")
    try:
        _above = sorted(passed, key=lambda r: -r["iptm_mean"])
    except NameError:
        _above = []
    if _above:
        print("\nabove the threshold at STEP 8, sorted by mean iPTM:")
        for _n, _r in enumerate(_above, 1):
            print("   %2d  %-16s %.3f" % (_n, _r["peptide"], _r["iptm_mean"]))
    _measured = [r["peptide"] for r in screened] if "screened" in dir() else []
    try:
        for _attempt_n in (1, 2):
            _answer = input("\npeptide - the number from the list, or a sequence "
                            "for one outside it: ").strip()
            # The number is the way without typos; a sequence is still
            # accepted, for a peptide that is not among the thirty.
            if _answer.isdigit() and _above and 1 <= int(_answer) <= len(_above):
                CANDIDATE = _above[int(_answer) - 1]["peptide"]
                print("   %s" % CANDIDATE)
                break
            CANDIDATE = _answer.upper()
            if not CANDIDATE or not _measured or CANDIDATE in _measured:
                break
            import difflib
            _close = difflib.get_close_matches(CANDIDATE, _measured, n=3, cutoff=0.7)
            print("\n   %s is not among the %d peptides STEP 8 measured."
                  % (CANDIDATE, len(_measured)))
            if _close:
                print("   did you mean %s ?" % ", ".join(_close))
            if _attempt_n == 1:
                print("   enter it again - a number, or the same sequence twice,")
                print("   which is taken as deliberate.")
            else:
                print("   entered twice: taken as deliberate.")
        CANDIDATE_DECIDED_BY = input("who decided (optional): ").strip()
        CANDIDATE_BASIS = input("on what basis (optional): ").strip()
    except (EOFError, OSError):
        print("\nno input available here: fill in the three constants at the top")
        print("of this cell and run it again.")

if not CANDIDATE:
    print("\nnothing entered: this stage computes nothing without a candidate,")
    print("because the 29 perturbations are perturbations of it.")
else:

    _measured = {r["peptide"] for r in screened} if "screened" in dir() else set()
    if _measured and CANDIDATE not in _measured:
        print("\n   %s is not among the %d peptides STEP 8 measured. That is"
              % (CANDIDATE, len(_measured)))
        print("   allowed - the decision is yours - but it is recorded as such.")
    print("\ncandidate   %s" % CANDIDATE)
    print("decided by  %s" % (CANDIDATE_DECIDED_BY or "NOT STATED"))
    print("basis       %s" % (CANDIDATE_BASIS or "NOT STATED"))
    if not CANDIDATE_DECIDED_BY or not CANDIDATE_BASIS:
        print("            (optional; the tables carry them as NOT STATED)")

    panel_applies = (CANDIDATE == PERTURBATIONS_OF)
    if not panel_applies:
        print("\n   The 29 perturbations and the position maps belong to %s."
              % PERTURBATIONS_OF)
        print("   %s is a different peptide: they are not its perturbations and are" % CANDIDATE)
        print("   left out. Only the alanine scan applies to it, and the effect size")
        print("   against the perturbation panel is not computed.")

    blocks = []
    if panel_applies:
        blocks += [(s, "perturbation: " + c, PERTURBATIONS_OF) for s, c in PERTURBATIONS]

    # one alanine at a time
    for i, wt in enumerate(CANDIDATE, 1):
        if wt == "A":
            continue
        role = ANCHORS.get(i, "not an anchor") if panel_applies else "position %d" % i
        blocks.append((substitute(CANDIDATE, [i]),
                       "alanine scan: %s%dA, %s" % (wt, i, role), CANDIDATE))

    if panel_applies:
        for pos, what in COMBINATIONS:
            blocks.append((substitute(CANDIDATE, pos),
                           "combination: %s, %s"
                           % ("+".join("%s%dA" % (CANDIDATE[p - 1], p) for p in pos), what),
                           CANDIDATE))
        # the W moved by one, composition unchanged
        w = CANDIDATE.find("W")
        if w > 0:
            blocks.append((CANDIDATE[:w - 1] + CANDIDATE[w] + CANDIDATE[w - 1] + CANDIDATE[w + 1:],
                           "geometry: W one residue earlier, same composition", CANDIDATE))
        if 0 <= w < len(CANDIDATE) - 1:
            blocks.append((CANDIDATE[:w] + CANDIDATE[w + 1] + CANDIDATE[w] + CANDIDATE[w + 2:],
                           "geometry: W one residue later, same composition", CANDIDATE))
        # the mirror on p53
        for p53pos, candpos, what in MIRROR:
            blocks.append((substitute(P53, [p53pos - P53_FIRST + 1]),
                           "p53 mirror: %s%dA of p53 <-> %s%dA of the candidate, %s"
                           % (P53[p53pos - P53_FIRST], p53pos,
                              CANDIDATE[candpos - 1], candpos, what), P53))
        for p53pos, candpos, what in MIRROR_PAIRS:
            blocks.append((substitute(P53, [p - P53_FIRST + 1 for p in p53pos]),
                           "p53 mirror: %s of p53 <-> %s of the candidate, %s"
                           % ("+".join("%s%dA" % (P53[p - P53_FIRST], p) for p in p53pos),
                              "+".join("%s%dA" % (CANDIDATE[p - 1], p) for p in candpos), what),
                           P53))

    # the candidate and p53 unmutated: the two references the deltas are taken from
    blocks += [(CANDIDATE, "reference: the candidate, unmutated", CANDIDATE),
               (P53, "reference: p53(17-29), unmutated", P53)]

    seen, jobs, manifest = set(), [], []
    for seq, role, parent in blocks:
        if seq in seen:
            continue
        seen.add(seq)
        # The category comes first in the name, so that sorting the AlphaFold 3
        # Server history by name groups them: ALANINE, GEOMETRY, MIRROR,
        # PERTURB, REF. Without it the forty-nine arrive as one list.
        _cat = ("REF" if role.startswith("reference") else
                "ALANINE" if role.startswith("alanine") else
                "ALANINE_COMBO" if role.startswith("combination") else
                "GEOMETRY" if role.startswith("geometry") else
                "MIRROR" if role.startswith("p53 mirror") else
                "PERTURB")
        jobs.append(af3_job("%s_%s_%s" % (TARGET, _cat, seq), seq))
        manifest.append({"peptide": seq, "length": len(seq), "role": role, "parent": parent})

    p_jobs = os.path.join(OUT_CAND, "af3_jobs_candidate_mechanism.json")
    p_man = os.path.join(OUT_CAND, "candidate_mechanism_manifest.csv")
    with open(p_jobs, "w") as f:
        json.dump(jobs, f, indent=1)
    with open(p_man, "w", newline="") as f:
        w_ = csv.DictWriter(f, fieldnames=list(manifest[0].keys()))
        w_.writeheader(); w_.writerows(manifest)

    print("\n%d jobs" % len(jobs))
    print("written:\n   %s\n   %s" % (p_jobs, p_man))
    _pool9, _ = complexes_by_sequence(verbose=False)
    _to_run9 = [s for s, _r, _p in blocks if s not in _pool9]
    if _to_run9:
        print("\n%d of these have not been co-folded yet. Run them, put the"
              % len(set(_to_run9)))
        print("archives in '%s' and run this cell again." % EXTERNAL)
    deliver(p_jobs, needed=bool(_to_run9), n_jobs=len(jobs))

    # ── read back ──────────────────────────
    # The 29 perturbations and the two references are the benchmark of the
    # effect size, so they are required.
    _required = sorted({s for s, _r, _p in blocks
                        if _r.startswith(("perturbation", "reference"))})
    _extra = len({s for s, _r, _p in blocks}) - len(_required)
    print("\nOf these %d, the %d perturbations and the %d references have to be in"
          % (len(jobs), len(PERTURBATIONS), 2))
    print("an archive before this stage computes anything: the effect size is taken")
    print("on all of them or on none. The other %d are reported as far as they have"
          % _extra)
    print("been co-folded.")
    require(_required, "29 perturbations and 2 references",
        "af3_results_candidate_mechanism.zip")
    pool, dupes = complexes_by_sequence(verbose=False)

    want = {m["peptide"]: m for m in manifest}
    rows = []
    for seq, m in want.items():
        c = pool.get(seq)
        if c is None:
            continue
        rows.append({"peptide": seq, "role": m["role"], "parent": m["parent"],
                     "iptm_mean": st.mean(c["iptm"]), "iptm_sd": st.stdev(c["iptm"]),
                     "receptor_len": c["receptor_len"], "archive": c["archive"],
                     "job": c["job"]})

    absent = [s for s in want if s not in pool]
    if absent:
        print("\n   %d of %d not yet in '%s': %s"
              % (len(absent), len(want), EXTERNAL,
                 ", ".join(absent[:6]) + (" ..." if len(absent) > 6 else "")))
    odd = [r for r in rows if r["receptor_len"] != len(RECEPTOR)]
    if odd:
        print("   ANOMALY: complexes on another construct: %s"
              % ", ".join("%s (%d)" % (r["peptide"], r["receptor_len"]) for r in odd))
    if rows:
        ref = {r["parent"]: r["iptm_mean"] for r in rows if r["role"].startswith("reference")}

        # The deltas are taken from the candidate as the screen measured it, not
        # from whichever run of it happens to be in the panel archives.
        screen_val = None
        try:
            screen_val = next(r["iptm_mean"] for r in screened if r["peptide"] == CANDIDATE)
        except (NameError, StopIteration):
            pass
        if screen_val is not None:
            if CANDIDATE in ref and abs(ref[CANDIDATE] - screen_val) > 1e-9:
                print("\nthe candidate has two runs: %.3f in the screen of STEP 8 and %.3f in"
                      % (screen_val, ref[CANDIDATE]))
                print("the panel archives, %.3f apart. The deltas below are taken from the"
                      % abs(ref[CANDIDATE] - screen_val))
                print("screening run, the one the rest of the work quotes.")
            ref[CANDIDATE] = screen_val
        elif CANDIDATE in ref:
            print("\nSTEP 8 has not been run in this session: the candidate's reference is")
            print("its run inside the panel archives, %.3f." % ref[CANDIDATE])

        def show(title, keep):
            sel = sorted([r for r in rows if keep(r)], key=lambda r: -r["iptm_mean"])
            if not sel:
                return
            print("\n%s" % title)
            print("  %-16s%8s%8s%9s  %s" % ("peptide", "iPTM", "SD", "delta", "role"))
            for r in sel:
                b = ref.get(r["parent"])
                d = "" if b is None else "%+9.3f" % (r["iptm_mean"] - b)
                print("  %-16s%8.3f%8.4f%9s  %s"
                      % (r["peptide"], r["iptm_mean"], r["iptm_sd"], d, r["role"]))

        show("References", lambda r: r["role"].startswith("reference"))
        show("Perturbation panel", lambda r: r["role"].startswith("perturbation"))
        show("Alanine scan of the candidate, and the combinations",
             lambda r: r["role"].startswith(("alanine", "combination", "geometry")))
        show("The p53 mirror", lambda r: r["role"].startswith("p53 mirror"))

        # ── effect size: the binders of STEP 1 against the 29 ───────────────
        pert = [r["iptm_mean"] for r in rows if r["role"].startswith("perturbation")]
        binders = (CALIBRATION or {}).get("binders")
        if pert and binders and len(binders) > 1 and len(pert) > 1:
            es = effect_size(binders, pert)
            gap = min(binders) - max(pert)
            print("\n%d binders with a published Kd against %d perturbed sequences"
                  % (es["n1"], es["n2"]))
            print("   df-weighted pooled SD   %.4f" % es["pooled_sd"])
            print("   Cohen's d               %.2f" % es["d"])
            print("   Hedges' g               %.2f     (J = %.5f)" % (es["g"], es["J"]))
            print("   95%% CI                  %.2f - %.2f" % (es["lo"], es["hi"]))
            print("   minimum gap             %+.3f" % gap)
            print("\n   With n = %d against n = %d this is a descriptive summary, not a"
                  % (es["n1"], es["n2"]))
            print("   basis for formal inference. It measures that the score falls on")
            print("   deliberately perturbed sequences; binders against non-binders is")
            print("   what STEP 1 measures.")

        # ── the mirror, side by side ────────────────────────────────────────
        # A pair is complete only when both sides are in an archive: the p53
        # variant and the variant of the candidate at the matching position.
        # The measurement is the pair, so the table needs one complete pair.
        by_seq = {r["peptide"]: r for r in rows}
        todo = [((p53pos,), (candpos,), what) for p53pos, candpos, what in MIRROR] + \
               [(a, b, w2) for a, b, w2 in MIRROR_PAIRS]
        complete, incomplete = [], []
        if CANDIDATE in ref and P53 in ref:
            for p53pos, candpos, what in todo:
                sp = substitute(P53, [p - P53_FIRST + 1 for p in p53pos])
                sc = substitute(CANDIDATE, list(candpos))
                rp, rc = by_seq.get(sp), by_seq.get(sc)
                lbl = "+".join("%s%dA" % (P53[p - P53_FIRST], p) for p in p53pos)
                if rp and rc:
                    complete.append(("%s  (%s)" % (lbl, what),
                                     rp["iptm_mean"] - ref[P53],
                                     rc["iptm_mean"] - ref[CANDIDATE]))
                else:
                    incomplete.append((lbl, "p53" if not rp else "candidate"))

        if complete:
            print("\nThe mirror, pair by pair. Same sign and similar size means the")
            print("two peptides lose confidence at the same positions.")
            print("  %-46s%11s%11s" % ("position", "p53", "candidate"))
            for lbl, dp, dc in complete:
                print("  %-46s%+11.3f%+11.3f" % (lbl, dp, dc))
        if incomplete:
            print("\n%d of the %d mirror pairs are not complete: one side of each"
                  % (len(incomplete), len(todo)))
            print("has not been co-folded. The missing side, pair by pair:")
            for lbl, _side in incomplete:
                print("   %-14s %s" % (lbl, _side))
        if not complete:
            print("\nNo pair is complete, so nothing is reported here: one side of a")
            print("pair on its own does not measure whether the two peptides lose")
            print("confidence at the same position.")


        t = os.path.join(OUT_CAND, "table_candidate_mechanism.csv")
        with open(t, "w", newline="") as f:
            w_ = csv.writer(f)
            w_.writerow(["peptide", "role", "parent", "iptm_mean", "iptm_sd",
                         "delta_vs_parent", "receptor_len", "archive", "job"])
            for r in sorted(rows, key=lambda r: (r["parent"], -r["iptm_mean"])):
                b = ref.get(r["parent"])
                w_.writerow([r["peptide"], r["role"], r["parent"], round(r["iptm_mean"], 4),
                             round(r["iptm_sd"], 4),
                             "" if b is None else round(r["iptm_mean"] - b, 4),
                             r["receptor_len"], r["archive"], r["job"]])
        print("\nwritten: %s" % t)

## STEP 10 — 3 questions about the method

Three blocks, on the pipeline and not on the candidate. Each writes its own jobs
and reads its own archive, and none depends on the others.

| | question | peptides |
|---|---|---|
| 1 | does the ESM-2 ranking do anything? | the first 30 of the canonical set against the last 30; the head is already co-folded at STEP 8 and is not run again |
| 2 | is the Phe–Trp pair arbitrary? | 20 at the same i/i+4 spacing with a different pair, none of them carrying `F-x(3)-W` |
| 3 | does the method work with no motif at all? | the first 10 of the whole filtered pool ranked by ESM-2 alone |

The 20 and the 10 are written out in the cell, each with the rank it holds in
the pool it was drawn from: the whole filtered library ranked by ESM-2 with no
motif applied. Block 3 claims no rate: 10 peptides give an interval too wide to
mean anything.

In [ ]:
# ============================================================
# STEP 10 - METHOD VALIDATION
# ============================================================
# Three questions, three directories: each block is its own entry in the
# deposit.
OUT_RANK = os.path.join(OUT_ROOT, "f_rank_test")
OUT_CHEM = os.path.join(OUT_ROOT, "g_alternative_chemistry")
OUT_GEN = os.path.join(OUT_ROOT, "h_generalisability")
for _d in (OUT_RANK, OUT_CHEM, OUT_GEN):
    os.makedirs(_d, exist_ok=True)
DIR_OF = {"rank_test": OUT_RANK, "alternative_chemistry": OUT_CHEM,
          "generalisability": OUT_GEN}

N_ENDS = 30        # how many at each end of the ranking
N_ALT = 10         # how many per alternative motif definition
N_GEN = 10         # how many from the whole-pool ranking

CANONICAL = MOTIF                # the motif declared once, at STEP 3
# For the no-motif test a peptide is excluded if it carries the motif, and the
# stronger claim is reported separately: whether it contains no tryptophan at
# all, in which case no form of the pharmacophore could have selected it.


def write_jobs(name, items):
    """items: list of (peptide, dict of manifest columns)"""
    jobs = [af3_job("%s_%s_%s" % (TARGET, name.upper(), p), p) for p, _ in items]
    _d = DIR_OF[name]
    pj = os.path.join(_d, "af3_jobs_%s.json" % name)
    pm = os.path.join(_d, "%s_manifest.csv" % name)
    with open(pj, "w") as f:
        json.dump(jobs, f, indent=1)
    rows = [dict([("peptide", p)] + list(d.items())) for p, d in items]
    with open(pm, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader(); w.writerows(rows)
    _absent = [p for p, _d in items if p not in pool]
    deliver(pj, needed=bool(_absent), n_jobs=len(jobs))
    return pj, rows


def summarise(label, values, threshold):
    n = len(values)
    k = sum(1 for v in values if v >= threshold)
    print("   %-24s n=%-3d mean %.3f   %d/%d >= %.2f"
          % (label, n, st.mean(values), k, n, threshold))
    return k, n


# Nothing is required here: each question reports how much of its own set has
# been co-folded.
pool, dupes = complexes_by_sequence(verbose=False)
ranked = sorted(candidates, key=lambda e: -e.get("esm2_similarity", 0))
rank_of = {e["peptide"]: i + 1 for i, e in enumerate(ranked)}

# ═══════════════════════════════════════════ 1. does the ranking do anything?
print("=" * 70)
print("1. THE RANKING: first %d against last %d of %d canonical candidates"
      % (N_ENDS, N_ENDS, len(ranked)))

head = ranked[:N_ENDS]
tail = ranked[-N_ENDS:]
head_done = {}
try:
    head_done = {r["peptide"]: r["iptm_mean"] for r in screened}
except NameError:
    pass

# The job file carries the full tail. What is already in an archive is reported
# below, so it need not be run again.
missing_head = [e for e in head if e["peptide"] not in head_done]
items = [(e["peptide"], {"esm_rank": rank_of[e["peptide"]], "group": "tail",
                         "esm2_similarity": e.get("esm2_similarity"),
                         "spacing": e.get("spacing")}) for e in tail]
items += [(e["peptide"], {"esm_rank": rank_of[e["peptide"]], "group": "head",
                          "esm2_similarity": e.get("esm2_similarity"),
                          "spacing": e.get("spacing")}) for e in missing_head]
pj, _ = write_jobs("rank_test", items)
_already = sum(1 for e in tail if e["peptide"] in pool)
print("   %d jobs written: the %d of the tail%s"
      % (len(items), len(tail),
         "" if not missing_head else ", and %d of the head that STEP 8 has not measured"
         % len(missing_head)))
if _already:
    print("   %d of the tail are already in an archive and need not be re-run"
          % _already)
print("   -> %s" % pj)

# If part of the tail is missing, its archive is asked for here.
_absent = [e["peptide"] for e in tail if e["peptide"] not in pool]
if _absent:
    take(_absent, "the tail of the ranking", expected_archive(pj))
    pool, dupes = complexes_by_sequence(verbose=False)

vals = {"head": [], "tail": []}
rows = []
for group, group_items in (("head", head), ("tail", tail)):
    for e in group_items:
        p = e["peptide"]
        v = head_done.get(p) if group == "head" else None
        src = "read at STEP 8"
        if v is None:
            c = pool.get(p)
            if c is None:
                continue
            v = st.mean(c["iptm"]); src = c["archive"]
        vals[group].append(v)
        rows.append({"peptide": p, "esm_rank": rank_of[p], "group": group,
                     "esm2_similarity": e.get("esm2_similarity"), "iptm_mean": round(v, 4),
                     "above_threshold": "yes" if v >= THRESHOLD else "no", "archive": src})

if len(vals["head"]) > 1 and len(vals["tail"]) > 1:
    print("\n   available: head %d of %d, tail %d of %d"
          % (len(vals["head"]), N_ENDS, len(vals["tail"]), N_ENDS))
    if len(vals["head"]) < N_ENDS or len(vals["tail"]) < N_ENDS:
        print("   What follows is computed on what is available, not on %d against %d."
              % (N_ENDS, N_ENDS))
    kh, nh = summarise("head (ranks 1-%d)" % N_ENDS, vals["head"], THRESHOLD)
    kt, nt = summarise("tail (last %d)" % N_ENDS, vals["tail"], THRESHOLD)
    es = effect_size(vals["head"], vals["tail"])
    gap = min(vals["head"]) - max(vals["tail"])
    print("\n   Cohen's d   %.2f      Hedges' g   %.2f" % (es["d"], es["g"]))
    print("   95%% CI      %.2f - %.2f" % (es["lo"], es["hi"]))
    print("   minimum gap %+.3f   %s"
          % (gap, "the two ends do not overlap" if gap > 0 else
                  "THE ENDS OVERLAP: the ranking shifts the distribution, it does not "
                  "delimit it"))
    with open(os.path.join(OUT_RANK, "table_rank_test.csv"), "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader(); w.writerows(sorted(rows, key=lambda r: r["esm_rank"]))
    print("   -> %s" % os.path.join(OUT_RANK, "table_rank_test.csv"))
else:
    have = len(vals["head"]) + len(vals["tail"])
    print("\n   %d of %d complexes available: run the jobs above and put the archive"
          % (have, 2 * N_ENDS))
    print("   in '%s', then run this cell again." % EXTERNAL)

# ═══════════════════════════════════════════ 2. is the definition arbitrary?
print("\n" + "=" * 70)
print("2. THE MOTIF DEFINITION: the alternative chemistries")

# Drawn once from the whole filtered pool ranked by ESM-2 with no motif
# applied; the rank each holds in that pool is kept beside it.
ALT_CHEMISTRY = [
    ("LSENDWLDLFK",   "ARO", 48),  ("DEWLNPFSLK",    "ARO", 71),
    ("TSLLEWFANEYK",  "ARO", 104), ("SYQDLFEGLLNK",  "ARO", 148),
    ("ELNWSAQFTDLEK", "ARO", 156), ("LPEWLENFK",     "ARO", 290),
    ("FLEDFASNFELTK", "ARO", 327), ("LESLTLKGFPDDF", "ARO", 329),
    ("AFMETYLPSFNEK", "ARO", 356), ("IDWLPNYSELK",   "ARO", 359),
    ("ESHPLLLEWFEK",  "MIX", 39),  ("DLTSWFEELR",    "MIX", 45),
    ("EELSTYWPILADK", "MIX", 51),  ("DLSSLWNLNLEGK", "MIX", 60),
    ("SALEDYFTLSPK",  "MIX", 65),  ("TSLDEFWK",      "MIX", 66),
    ("LSEEILTDWNHLK", "MIX", 72),  ("NSFENLLEPFLK",  "MIX", 73),
    ("DTNLEILLESFSK", "MIX", 76),  ("ESEALLPSWINK",  "MIX", 79),
]
ALT_NAME = {"ARO": "aromatic pair, [FWY]-x(3)-[FWY]",
            "MIX": "hydrophobic-aromatic, [FWYLIM]-x(3)-[FWY]"}

print("   %d peptides, geometry held at i/i+4 and the chemistry varied, none"
      % len(ALT_CHEMISTRY))
print("   carrying F-x(3)-W. Drawn from the whole-pool ranking with no motif")
print("   applied; their rank in that pool is reported.")

_items = [(p, {"class": cls, "definition": ALT_NAME[cls], "pool_rank": r})
          for p, cls, r in ALT_CHEMISTRY]
_pj, _ = write_jobs("alternative_chemistry", _items)
print("   -> %s" % _pj)

_absent = [p for p, _c, _r in ALT_CHEMISTRY if p not in pool]
if _absent:
    take(_absent, "the alternative chemistries", expected_archive(_pj))
    pool, dupes = complexes_by_sequence(verbose=False)

_rows = []
for p, cls, r in ALT_CHEMISTRY:
    c = pool.get(p)
    if not c:
        continue
    m = st.mean(c["iptm"])
    _rows.append({"peptide": p, "class": cls, "definition": ALT_NAME[cls],
                  "pool_rank": r, "iptm_mean": round(m, 4),
                  "iptm_sd": round(st.stdev(c["iptm"]), 4) if len(c["iptm"]) > 1 else 0.0,
                  "n_models": len(c["iptm"]), "above_threshold":
                  "yes" if m >= THRESHOLD else "no",
                  "archive": c["archive"], "job": c["job"]})

if not _rows:
    print("   none co-folded yet: run the jobs above and load the archive here.")
else:
    print("\n   %-16s%-6s%9s%9s%8s" % ("peptide", "class", "pool", "iPTM", ">=thr"))
    for r_ in sorted(_rows, key=lambda x: -x["iptm_mean"]):
        print("   %-16s%-6s%9d%9.3f%8s"
              % (r_["peptide"], r_["class"], r_["pool_rank"], r_["iptm_mean"],
                 r_["above_threshold"]))
    t = os.path.join(OUT_CHEM, "table_alternative_chemistry.csv")
    with open(t, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(_rows[0].keys()))
        w.writeheader(); w.writerows(_rows)
    print("   -> %s" % t)
    for cls in ("ARO", "MIX"):
        v = [r_["iptm_mean"] for r_ in _rows if r_["class"] == cls]
        if v:
            summarise(ALT_NAME[cls], v, THRESHOLD)
    if vals["head"]:
        summarise("canonical F-x(3)-W, the head of STEP 8", vals["head"], THRESHOLD)
    print("\n   Against the canonical set above: if the alternative pairs reach")
    print("   the same confidence, the choice of Phe and Trp is arbitrary. Same")
    print("   receptor, same seed, same five models on both sides.")

print("\n" + "=" * 70)
print("3. GENERALISABILITY: the top of the ranking with no motif filter at all")

# The first ten of the whole filtered pool ranked by ESM-2 alone, with no
# motif of any kind. Their rank in that pool is kept.
NO_MOTIF = [
    ("SELEDLLNFPK", 1),   ("EPETLSNFIDLLK", 2),  ("EVLSDFESLLPK", 3),
    ("QLATSLDEFWK", 4),   ("ANSFENLLEPFLK", 5),  ("SFNELVSLWEK", 6),
    ("EILSDFESLLPK", 7),  ("FDSFTELLNK", 8),     ("HLPELLSDWK", 9),
    ("LSVLNEFLPEDK", 10),
]
_rx_can = _re_module.compile(CANONICAL)
print("   %d peptides, the top of the pool by ESM-2 alone. %d of them carry no"
      % (len(NO_MOTIF), sum(1 for p, _ in NO_MOTIF if "W" not in p)))
print("   tryptophan at all, so no form of the pharmacophore could select them.")

_items3 = [(p, {"pool_rank": r, "has_tryptophan": "yes" if "W" in p else "no",
                "carries_motif": "yes" if _rx_can.search(p) else "no"})
           for p, r in NO_MOTIF]
_pj3, _ = write_jobs("generalisability", _items3)
print("   -> %s" % _pj3)

_absent3 = [p for p, _r in NO_MOTIF if p not in pool]
if _absent3:
    take(_absent3, "the no-motif peptides", expected_archive(_pj3))
    pool, dupes = complexes_by_sequence(verbose=False)

_rows3 = []
for p, r in NO_MOTIF:
    c = pool.get(p)
    if not c:
        continue
    m = st.mean(c["iptm"])
    _rows3.append({"peptide": p, "pool_rank": r,
                   "has_tryptophan": "yes" if "W" in p else "no",
                   "carries_motif": "yes" if _rx_can.search(p) else "no",
                   "iptm_mean": round(m, 4),
                   "iptm_sd": round(st.stdev(c["iptm"]), 4) if len(c["iptm"]) > 1 else 0.0,
                   "n_models": len(c["iptm"]),
                   "above_threshold": "yes" if m >= THRESHOLD else "no",
                   "archive": c["archive"], "job": c["job"]})

if not _rows3:
    print("   none co-folded yet: run the jobs above and load the archive here.")
else:
    print("\n   %-16s%7s%6s%9s%8s" % ("peptide", "pool", "W", "iPTM", ">=thr"))
    for r_ in sorted(_rows3, key=lambda x: -x["iptm_mean"])[:12]:
        print("   %-16s%7d%6s%9.3f%8s"
              % (r_["peptide"], r_["pool_rank"], r_["has_tryptophan"],
                 r_["iptm_mean"], r_["above_threshold"]))
    t3 = os.path.join(OUT_GEN, "table_generalisability.csv")
    with open(t3, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(_rows3[0].keys()))
        w.writeheader(); w.writerows(_rows3)
    print("   -> %s" % t3)
    summarise("no motif, ESM-2 rank only", [r_["iptm_mean"] for r_ in _rows3], THRESHOLD)
    _best = max(_rows3, key=lambda x: x["iptm_mean"])
    print("\n   Highest: %s at %.3f, pool rank %d, tryptophan: %s."
          % (_best["peptide"], _best["iptm_mean"], _best["pool_rank"],
             _best["has_tryptophan"]))
    print("   A route without the pharmacophore therefore exists and produces")
    print("   high-confidence complexes. On %d peptides that is an existence"
          % len(_rows3))
    print("   result: no rate is claimed.")

print("\n" + "=" * 70)
print("What stays outside this notebook: the molecular dynamics, the MM-PBSA,")
print("and the three other structure predictors - Chai-1, Protenix and EvoEF2.")
print("Each needs hours of GPU or a separate program, and each is deposited with")
print("its own files.")

## Export

Run last. It collects into one zip the candidate files, the lettered folder of
every stage that ran, and the AlphaFold 3 archives those stages read — each one
beside the job file it answers, so the zip carries the questions and the answers
together. What it leaves out, with sizes, is listed inside
`00_WHAT_IS_NOT_HERE.txt`.

The 6 job files do not wait for this cell: each stage hands its own over as soon
as it writes it.

In [ ]:
# ============================================================
# EXPORT - THE CANDIDATE FILES
# ============================================================
import os
import csv
import json
import glob
import shutil
import zipfile

# The candidate set as one CSV. After a runtime restart "candidates" is gone,
# so the JSON that STEP 7 left in RESULTS_DIR is read back instead.
try:
    _candidates = candidates
    _read_from = "memory"
except NameError:
    _hits = sorted(glob.glob(os.path.join(RESULTS_DIR, "b_candidates",
                                      "candidate_set_*.json")))
    if not _hits:
        _candidates, _read_from = [], "nothing found"
    else:
        with open(_hits[-1]) as _fh:
            _candidates = json.load(_fh)
        _read_from = os.path.basename(_hits[-1])

rows = [{
    "rank_in_set": _e.get("rank_in_set"),
    "peptide": _e.get("peptide"),
    "length": _e.get("length"),
    "esm2_similarity": _e.get("esm2_similarity"),
    "spacing": _e.get("spacing"),
    "pharmacophore": _e.get("pharmacophore"),
    "pharmacophore_match": _e.get("pharmacophore_match"),
    "species": _e.get("source"),
    "all_species": _e.get("sources"),
    "protein_id": _e.get("protein_id"),
    "protein_name": _e.get("protein_name"),
    "position": _e.get("position"),
    "enzyme": _e.get("enzyme"),
    "n_enzymes": _e.get("n_enzymes"),
} for _e in _candidates]

# Next to the JSON it summarises, in the same directory.
PH_CSV = os.path.join(RESULTS_DIR, "b_candidates", "candidate_set.csv")
os.makedirs(os.path.dirname(PH_CSV), exist_ok=True)
if rows:
    with open(PH_CSV, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0]))
        w.writeheader()
        w.writerows(rows)
    print(f"candidate_set.csv - {len(rows):,} rows (from {_read_from})")
    _sp = len({r["species"] for r in rows})
    _lead = any(r["peptide"] == "SPAFESTWDILK" for r in rows)
    print(f"   {len(rows):,} candidates | {_sp} species | SPAFESTWDILK present: {_lead}")
else:
    print("no candidate found: STEP 7 has not been run.")

print(f"\nfile in {RESULTS_DIR}:")
# What the archive leaves behind: the intermediate dumps. They are rebuilt by
# re-running STEPs 4 to 6 and no reported number is read from them directly.
# A .partial file is what an interrupted digestion leaves behind.
SCRATCH = ("peptides_hydrolysis.csv", "peptides_prefiltered.csv",
           "esm2_checkpoint.json")
# esm2_partial_<target>.json carries the target in its name, so it is matched by
# prefix.
SCRATCH_PREFIXES = ("esm2_partial_",)
SCRATCH_EXT = (".db", ".sqlite", ".sqlite3", ".zip", ".partial")

_all_files = sorted(glob.glob(os.path.join(RESULTS_DIR, "*")))
_kept, _left = [], []
for _f in _all_files:
    _b = os.path.basename(_f)
    _skip = (_b in SCRATCH or _b.lower().endswith(SCRATCH_EXT)
             or _b.startswith(SCRATCH_PREFIXES))
    (_left if _skip else _kept).append(_f)
    _kb = sum(os.path.getsize(os.path.join(_r, _n))
              for _r, _d, _fs in os.walk(_f) for _n in _fs) / 1024 \
        if os.path.isdir(_f) else os.path.getsize(_f) / 1024
    print(f"   {'-' if _skip else ' '} {_b:<50}{_kb:>12,.0f} KB")

ARCHIVE = os.path.join(BASE_DIR, "PepVeg_candidates_" + os.path.basename(RESULTS_DIR))
with zipfile.ZipFile(ARCHIVE + ".zip", "w", zipfile.ZIP_DEFLATED) as _z:
    for _f in _kept:
        if os.path.isdir(_f):
            for _r, _d, _fs in os.walk(_f):
                for _n in _fs:
                    _p = os.path.join(_r, _n)
                    _z.write(_p, os.path.relpath(_p, RESULTS_DIR))
        else:
            _z.write(_f, os.path.basename(_f))

    # The AF3 archives sit in af3_archives inside RESULTS_DIR, so the walk
    # above already took them. This only reports which ones are there.
    _arch_in = [(os.path.relpath(os.path.join(EXTERNAL, s_["archive"]), RESULTS_DIR),
                 os.path.getsize(os.path.join(EXTERNAL, s_["archive"])) / 1e6)
                for s_ in EXCHANGES
                if os.path.exists(os.path.join(EXTERNAL, s_["archive"]))]
    _arch_out = [s_["archive"] for s_ in EXCHANGES
                 if not os.path.exists(os.path.join(EXTERNAL, s_["archive"]))]

    _z.writestr("00_WHAT_IS_NOT_HERE.txt",
                "Left out of this archive, on purpose:\n\n"
                + "".join("  %-34s %10.1f MB\n"
                          % (os.path.basename(_f), os.path.getsize(_f) / 1e6)
                          for _f in _left if os.path.isfile(_f))
                + "\nThese are the intermediate dumps of the hydrolysis and of the\n"
                  "physicochemical filter. They are regenerated by re-running\n"
                  "STEPs 4 to 6, and no reported number is read from them\n"
                  "directly.\n"
                + ("" if not _arch_out else
                   "\nThese AlphaFold 3 archives were never loaded in this run, so\n"
                   "the stages that read them computed nothing:\n"
                   + "".join("  %s\n" % a for a in _arch_out)))

_mb_kept = os.path.getsize(ARCHIVE + ".zip") / 1e6
_mb_left = sum(os.path.getsize(_f) for _f in _left if os.path.isfile(_f)) / 1e6
print(f"\narchive: {ARCHIVE}.zip  ({_mb_kept:.1f} MB)")
if _arch_in:
    print(f"   with the {len(_arch_in)} AF3 archive(s) it read, one per stage:")
    for _d, _mb in _arch_in:
        print(f"      {_d:<56}{_mb:>7.1f} MB")
if _arch_out:
    print(f"   {len(_arch_out)} AF3 archive(s) were never loaded: "
          f"{', '.join(_arch_out)}")
print(f"left in {RESULTS_DIR}: {_mb_left:.0f} MB of intermediate dumps, listed")
print("inside the archive in 00_WHAT_IS_NOT_HERE.txt and regenerated by")
print("re-running STEPs 3 to 6.")

try:
    from google.colab import files
    files.download(ARCHIVE + ".zip")
except Exception as _e:
    print(f"\nautomatic download unavailable: take {ARCHIVE}.zip from the File "
          f"panel on the left")

# The six exchanges with the server, and whether each still has something to
# run. Printed last so it stays on screen.
exchange_summary()
